# Sample Comparison: All Methods (4×4)

Single-sample comparison on the Open718 paper-side test field:

```text
Paper/EBSD_SR_Nature_v3/Open_718_Test_hr_x_block_0.npy
```

This file is HR-only and stores quaternions as scalar-last `[x,y,z,w]`. The scalar position is checked directly from the component magnitudes. The active/passive side is fixed by the OCRP source/API and repo EBSD convention: stored EBSD maps are treated as passive and the encoder internally conjugates them to active. The notebook therefore first reorders the file to passive scalar-first `[w,x,y,z]`, then crops it to the aligned top-left `300×388` region and derives the 4× LR input by stride-4 sampling, producing a `75×97` LR map.

All IPF rows in this comparison are rendered from passive scalar-first `[w,x,y,z]` quaternions. This is deliberate: it matches the paper-side reference `Open_718_Test_hr_x_block_0_ipf_z.png`. Active-conjugated quaternions are not used for IPF visualization.

The comparison shows:

- paper learned baselines as columns: OCRP, EDSR, QEDSR, Q-RBSA-adapted, HAN, RCAN, SAN, and optional Atindama inpainting;
- classical/geometric methods as context columns;
- IPF `X / Y / Z` as separate rows;
- full-map boundary checks;
- a large-blue-grain interior patch with intra-grain GROD/KAM/misorientation metrics.


In [1]:
# ── GPU Selection: pick the GPU with the most free memory ──────────────────
import os
import subprocess

os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
SELECTED_CUDA_DEVICE_INDEX = None


def _pick_most_free_gpu():
    try:
        result = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=index,memory.free",
                "--format=csv,noheader,nounits",
            ],
            check=True,
            capture_output=True,
            text=True,
        )
    except Exception as exc:
        print(f"GPU auto-select skipped: {exc}")
        return None

    rows = []
    for line in result.stdout.strip().splitlines():
        parts = [part.strip() for part in line.split(",")]
        if len(parts) < 2:
            continue
        idx_text, free_text = parts[:2]
        if idx_text.isdigit() and free_text.isdigit():
            rows.append((int(free_text), int(idx_text)))

    if not rows:
        print("GPU auto-select skipped: no GPUs reported by nvidia-smi")
        return None

    free_mb, gpu_idx = max(rows)
    print(f"Using GPU {gpu_idx} with {free_mb} MiB free")
    return gpu_idx


SELECTED_CUDA_DEVICE_INDEX = _pick_most_free_gpu()

if "torch" in globals() and SELECTED_CUDA_DEVICE_INDEX is not None and torch.cuda.is_available():
    try:
        torch.cuda.set_device(SELECTED_CUDA_DEVICE_INDEX)
        print(f"torch already imported; switched current device to cuda:{SELECTED_CUDA_DEVICE_INDEX}")
        print("Re-run Section 1 and later cells so they use the selected GPU explicitly.")
    except Exception as exc:
        print(f"Could not switch torch CUDA device: {exc}")


Using GPU 6 with 40326 MiB free


In [2]:
# ── Section 1: Setup ─────────────────────────────────────────────────────────
import os
import sys
import json
from pathlib import Path
from collections import OrderedDict

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/numba")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import uniform_filter, label as ndi_label

repo_root = Path.cwd()
if not (repo_root / "training").exists():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "training").exists():
            repo_root = p
            break
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

analysis_dir = str(repo_root / "analysis")
if analysis_dir not in sys.path:
    sys.path.insert(0, analysis_dir)

from training.config_utils import load_and_prepare_config
from inference import infer_iso_embedding_sr_attn as inf
from training.train_jangid_baseline import (
    build_model as build_jangid_model,
    to_chw as jangid_to_chw,
    to_hwc_numpy as jangid_to_hwc_numpy,
    conjugate_scalar_first_chw as jangid_conjugate_scalar_first_chw,
    forward_baseline_active_from_passive,
)
from training.train_atindama_inpainting import (
    load_authors_model,
    passive_quaternion_to_normalized_zxz,
    normalized_zxz_to_passive_quaternion,
    periodic_known_mask,
)
from inference.infer_atindama_inpainting import sanitize_prediction
from models.bicubic_f_interpolate_sr import QuaternionBicubicFInterpolateSR
from utils.symmetry_utils import resolve_symmetry
from utils import format_quaternions, reduce_to_fz_min_angle
from visualization.ipf_render import render_ipf_rgb

# Some imported paper-baseline helper modules force Matplotlib's non-interactive
# Agg backend for batch scripts.  In notebooks, restore the inline backend so
# plt.show() displays instead of warning that FigureCanvasAgg is non-interactive.
try:
    import matplotlib
    if "ipykernel" in sys.modules:
        matplotlib.use("module://matplotlib_inline.backend_inline", force=True)
        plt.switch_backend("module://matplotlib_inline.backend_inline")
except Exception as exc:
    print(f"Matplotlib inline backend restore skipped: {exc}")


def slugify_label(label: str) -> str:
    """Filesystem-safe label for saved notebook artifacts."""
    out = []
    for ch in str(label):
        if ch.isalnum():
            out.append(ch.lower())
        elif ch in ("+", "-"):
            out.append(ch)
        else:
            out.append("_")
    slug = "".join(out).strip("_")
    while "__" in slug:
        slug = slug.replace("__", "_")
    return slug or "unnamed"


def save_notebook_figure(fig, path: Path, *, dpi: int = 220):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
    print("saved:", path)
    return path


def save_rgb_panel(path: Path, rgb):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(rgb01_to_uint8(rgb)).save(path)
    return path


def save_mask_panel(path: Path, mask):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray((np.asarray(mask).astype(np.uint8) * 255)).save(path)
    return path

try:
    from skimage.metrics import structural_similarity as sk_ssim
    SKIMAGE_AVAILABLE = True
except ImportError:
    sk_ssim = None
    SKIMAGE_AVAILABLE = False

selected_gpu = globals().get("SELECTED_CUDA_DEVICE_INDEX", None)
if torch.cuda.is_available():
    if selected_gpu is None:
        selected_gpu = torch.cuda.current_device()
    device = torch.device(f"cuda:{int(selected_gpu)}")
    torch.cuda.set_device(device)
else:
    device = torch.device("cpu")
sym = resolve_symmetry("Oh")

print("repo_root:", repo_root)
print("device:", device)
if device.type == "cuda":
    print("cuda_device_name:", torch.cuda.get_device_name(device))
print("symmetry:", getattr(sym, "name", "Oh"))


/data/home/umang/Materials/Reynolds-QSR_paper/third_party/UCSB-Q-RBSA/mat_sci_torch_quats/quats.py:18: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647327489/work/torch/csrc/utils/tensor_new.cpp:278.)
  Q_arr = torch.Tensor([q1,qi,qj,qk])


repo_root: /data/home/umang/Materials/Reynolds-QSR_paper
device: cuda:6
cuda_device_name: NVIDIA A100-PCIE-40GB
symmetry: m-3m


In [3]:
# ── Section 2: Quaternion + metric helpers (matching metrics notebook) ─────
def normalize_quat(q, eps=1e-12):
    q = np.asarray(q, dtype=np.float64)
    if q.size == 0:
        return q
    if q.shape[-1] != 4:
        raise ValueError("Quaternion arrays must have last axis size 4")
    n = np.linalg.norm(q.reshape(-1, 4), axis=1).reshape(q.shape[:-1] + (1,))
    return q / (n + eps)


def quat_conjugate(q):
    q = np.asarray(q)
    return q * np.array([1.0, -1.0, -1.0, -1.0])


def xyzw_to_wxyz(q_xyzw):
    q = np.asarray(q_xyzw, dtype=np.float32)
    if q.shape[-1] != 4:
        raise ValueError(f"expected quaternion-last array, got {q.shape}")
    return np.concatenate([q[..., 3:4], q[..., :3]], axis=-1).astype(np.float32)


def infer_scalar_layout(q_raw, *, ratio_threshold=1.5):
    q = np.asarray(q_raw, dtype=np.float32)
    if q.shape[-1] != 4:
        raise ValueError(f"expected quaternion-last array, got {q.shape}")
    abs_mean = np.mean(np.abs(q.reshape(-1, 4)), axis=0)
    first, last = float(abs_mean[0]), float(abs_mean[-1])
    if first > ratio_threshold * max(last, 1e-12):
        return "wxyz"
    if last > ratio_threshold * max(first, 1e-12):
        return "xyzw"
    raise ValueError(
        "Cannot confidently infer scalar position from component magnitudes: "
        f"abs_mean={abs_mean}. Set the scalar layout explicitly."
    )


def raw_to_passive_wxyz(q_raw, scalar_layout="auto"):
    q = np.asarray(q_raw, dtype=np.float32)
    layout = infer_scalar_layout(q) if scalar_layout == "auto" else str(scalar_layout).lower()
    abs_mean = np.mean(np.abs(q.reshape(-1, 4)), axis=0)
    if layout == "xyzw":
        q_passive = xyzw_to_wxyz(q)
    elif layout == "wxyz":
        q_passive = q.astype(np.float32, copy=True)
    else:
        raise ValueError(f"scalar_layout must be 'auto', 'xyzw', or 'wxyz', got {scalar_layout!r}")
    return normalize_quat(q_passive).astype(np.float32), layout, abs_mean


def rgb01_to_uint8(rgb):
    return np.clip(np.asarray(rgb) * 255.0, 0, 255).round().astype(np.uint8)


def extract_reference_ipfz_map_crop(path):
    """Extract the colored map panel from the sidecar MATLAB/MTEX-style IPF-Z PNG."""
    rgb = np.asarray(Image.open(path).convert("RGB"))
    # The right side is the color key; the map panel is on the left.  Use only
    # the left half so the triangular key is never included in the crop.
    left_panel = rgb[:, : max(1, int(0.50 * rgb.shape[1]))]
    arr = left_panel.astype(np.float32) / 255.0
    saturation = arr.max(axis=-1) - arr.min(axis=-1)
    yy = np.indices(left_panel.shape[:2])[0]
    mask = (saturation > 0.08) & (yy > 80)
    ys, xs = np.where(mask)
    if ys.size == 0:
        raise RuntimeError(f"Could not find IPF map panel inside {path}")
    box = (int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1)
    crop = rgb[box[1]:box[3], box[0]:box[2]]
    return crop, box, tuple(int(x) for x in rgb.shape)


def quat_mul(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    if a.shape[-1] != 4 or b.shape[-1] != 4:
        raise ValueError("Input quaternions must have shape (..., 4)")
    w1, x1, y1, z1 = a[..., 0], a[..., 1], a[..., 2], a[..., 3]
    w2, x2, y2, z2 = b[..., 0], b[..., 1], b[..., 2], b[..., 3]
    w = w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2
    x = w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2
    y = w1 * y2 - x1 * z2 + y1 * w2 + z1 * x2
    z = w1 * z2 + x1 * y2 - y1 * x2 + z1 * w2
    return np.stack((w, x, y, z), axis=-1)


def crystallographic_misorientation(q_pred, q_gt, sym_quats=None, degrees=True):
    q_pred = np.asarray(q_pred, dtype=np.float64)
    q_gt = np.asarray(q_gt, dtype=np.float64)

    if q_pred.shape[-1] != 4 or q_gt.shape[-1] != 4:
        raise ValueError("Input quaternions must have last axis length 4")

    q_pred = normalize_quat(q_pred)
    q_gt = normalize_quat(q_gt)

    if sym_quats is None:
        sym_qs = None
    elif isinstance(sym_quats, str):
        sym_qs = np.asarray(resolve_symmetry(sym_quats).data, dtype=np.float64)
    elif hasattr(sym_quats, "data"):
        sym_qs = np.asarray(sym_quats.data, dtype=np.float64)
    elif hasattr(sym_quats, "to_quaternion"):
        sym_qs = np.asarray(sym_quats.to_quaternion(), dtype=np.float64)
    elif isinstance(sym_quats, np.ndarray):
        sym_qs = np.asarray(sym_quats, dtype=np.float64)
        if sym_qs.ndim != 2 or sym_qs.shape[1] != 4:
            raise ValueError("sym_quats numpy array must have shape (G, 4)")
    else:
        raise ValueError("Unknown sym_quats type")

    shape = q_pred.shape[:-1]
    qp = q_pred.reshape(-1, 4)
    qg = q_gt.reshape(-1, 4)

    if sym_qs is None:
        dots = np.abs(np.sum(qp * qg, axis=-1))
        dots = np.clip(dots, -1.0, 1.0)
        ang = 2.0 * np.arccos(dots)
        if degrees:
            ang = np.degrees(ang)
        return ang.reshape(shape)

    ops = sym_qs.copy()
    ops[:, 1:] *= -1.0
    gq = quat_mul(ops[:, None, :], qg[None, :, :])
    dots = np.abs(np.sum(gq * qp[None, :, :], axis=-1))
    dots = np.clip(dots, -1.0, 1.0)
    ang = 2.0 * np.arccos(dots)
    min_ang = np.min(ang, axis=0)
    if degrees:
        min_ang = np.degrees(min_ang)
    return min_ang.reshape(shape)


def misorientation_map(pred_q, gt_q, sym_quats=None, degrees=True):
    return crystallographic_misorientation(pred_q, gt_q, sym_quats=sym_quats, degrees=degrees)


def quaternion_mean(qs):
    qs = np.asarray(qs, dtype=np.float64).reshape(-1, 4)
    qs = normalize_quat(qs)
    M = np.dot(qs.T, qs)
    w, v = np.linalg.eigh(M)
    avg = v[:, np.argmax(w)]
    if avg[0] < 0:
        avg = -avg
    return normalize_quat(avg)


def compute_grod(ori_map, grain_labels=None, sym_ops=None, window_size=21):
    ori_map = np.asarray(ori_map, dtype=np.float64)
    H, W = ori_map.shape[:2]
    if grain_labels is not None:
        labels = np.asarray(grain_labels)
        grod = np.zeros((H, W), dtype=np.float64)
        grain_stats = {}
        for lab in np.unique(labels):
            mask = labels == lab
            if np.count_nonzero(mask) == 0:
                continue
            qs = ori_map[mask]
            mean_q = quaternion_mean(qs)
            grod_vals = crystallographic_misorientation(
                qs, np.tile(mean_q, (qs.shape[0], 1)), sym_quats=sym_ops, degrees=True
            )
            grod[mask] = grod_vals
            grain_stats[int(lab)] = {
                "mean": float(np.nanmean(grod_vals)),
                "max": float(np.nanmax(grod_vals)),
                "std": float(np.nanstd(grod_vals)),
                "count": int(qs.shape[0]),
            }
        return grod, grain_stats

    q = normalize_quat(ori_map)
    M_local = np.zeros((H, W, 4, 4), dtype=np.float64)
    for i in range(4):
        for j in range(i, 4):
            arr = q[..., i] * q[..., j]
            local = uniform_filter(arr, size=window_size, mode="reflect")
            M_local[..., i, j] = local
            M_local[..., j, i] = local
    M_flat = M_local.reshape(-1, 4, 4)
    w, v = np.linalg.eigh(M_flat)
    avg = v[:, :, -1]
    signs = np.where(avg[:, 0] < 0, -1.0, 1.0).reshape(-1, 1)
    avg = (avg * signs).reshape(H, W, 4)
    avg = normalize_quat(avg)
    grod = crystallographic_misorientation(ori_map, avg, sym_quats=sym_ops, degrees=True)
    stats = {
        "mean": float(np.nanmean(grod)),
        "max": float(np.nanmax(grod)),
        "std": float(np.nanstd(grod)),
    }
    return grod, stats


def compute_kam(ori_map, grain_labels=None, radius=1, sym_ops=None, ignore_threshold_deg=15.0, ref_patch=None):
    if ref_patch is not None:
        ori_map = ref_patch

    ori_map = np.asarray(ori_map, dtype=np.float64)
    H, W = ori_map.shape[:2]
    neighbors = []
    for dy in range(-radius, radius + 1):
        for dx in range(-radius, radius + 1):
            if dy == 0 and dx == 0:
                continue
            neighbors.append((dy, dx))

    vals = np.zeros((len(neighbors), H, W), dtype=np.float64)
    for k, (dy, dx) in enumerate(neighbors):
        src_y0 = max(0, -dy)
        src_y1 = H - max(0, dy)
        src_x0 = max(0, -dx)
        src_x1 = W - max(0, dx)
        dst_y0 = max(0, dy)
        dst_y1 = H - max(0, -dy)
        dst_x0 = max(0, dx)
        dst_x1 = W - max(0, -dx)

        shifted = np.zeros_like(ori_map)
        shifted[dst_y0:dst_y1, dst_x0:dst_x1] = ori_map[src_y0:src_y1, src_x0:src_x1]

        vm = np.zeros((H, W), dtype=bool)
        vm[dst_y0:dst_y1, dst_x0:dst_x1] = True
        if grain_labels is not None:
            same_grain = np.zeros((H, W), dtype=bool)
            same_grain[dst_y0:dst_y1, dst_x0:dst_x1] = (
                grain_labels[dst_y0:dst_y1, dst_x0:dst_x1]
                == grain_labels[src_y0:src_y1, src_x0:src_x1]
            )
            vm = vm & same_grain

        mis = np.full((H, W), np.nan, dtype=np.float64)
        if np.any(vm):
            mis_vals = crystallographic_misorientation(
                ori_map[vm], shifted[vm], sym_quats=sym_ops, degrees=True
            )
            mis[vm] = mis_vals
        if ignore_threshold_deg is not None:
            mis[mis > ignore_threshold_deg] = np.nan
        vals[k] = mis

    kam = np.nanmean(vals, axis=0)
    stats = {
        "mean": float(np.nanmean(kam)),
        "max": float(np.nanmax(kam)),
        "std": float(np.nanstd(kam)),
    }
    return kam, stats




def compute_boundary_mask(ori_map, *, sym_ops=None, threshold_deg=5.0, connectivity=4):
    ori_map = np.asarray(ori_map, dtype=np.float64)
    H, W = ori_map.shape[:2]
    neighbors = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    if connectivity == 8:
        neighbors += [(-1, -1), (-1, 1), (1, -1), (1, 1)]

    boundary = np.zeros((H, W), dtype=bool)
    for dy, dx in neighbors:
        src_y0 = max(0, -dy)
        src_y1 = H - max(0, dy)
        src_x0 = max(0, -dx)
        src_x1 = W - max(0, dx)
        dst_y0 = max(0, dy)
        dst_y1 = H - max(0, -dy)
        dst_x0 = max(0, dx)
        dst_x1 = W - max(0, -dx)

        shifted = np.zeros_like(ori_map)
        shifted[dst_y0:dst_y1, dst_x0:dst_x1] = ori_map[src_y0:src_y1, src_x0:src_x1]

        valid_mask = np.zeros((H, W), dtype=bool)
        valid_mask[dst_y0:dst_y1, dst_x0:dst_x1] = True

        mis = np.zeros((H, W), dtype=np.float64)
        if np.any(valid_mask):
            mis_vals = crystallographic_misorientation(
                ori_map[valid_mask],
                shifted[valid_mask],
                sym_quats=sym_ops,
                degrees=True,
            )
            mis[valid_mask] = mis_vals
        boundary |= mis > float(threshold_deg)

    return boundary


def boundary_f1_score(sr_boundary, hr_boundary):
    sr_boundary = np.asarray(sr_boundary, dtype=bool)
    hr_boundary = np.asarray(hr_boundary, dtype=bool)
    tp = int(np.sum(sr_boundary & hr_boundary))
    fp = int(np.sum(sr_boundary & ~hr_boundary))
    fn = int(np.sum(~sr_boundary & hr_boundary))
    denom = 2 * tp + fp + fn
    return float((2 * tp) / denom) if denom > 0 else float('nan')

def _resolve_symmetry_quats(sym_quats):
    if sym_quats is None:
        return None
    if isinstance(sym_quats, str):
        return np.asarray(resolve_symmetry(sym_quats).data, dtype=np.float64)
    if hasattr(sym_quats, "data"):
        return np.asarray(sym_quats.data, dtype=np.float64)
    if hasattr(sym_quats, "to_quaternion"):
        return np.asarray(sym_quats.to_quaternion(), dtype=np.float64)
    arr = np.asarray(sym_quats, dtype=np.float64)
    if arr.ndim != 2 or arr.shape[1] != 4:
        raise ValueError("sym_quats numpy array must have shape (G, 4)")
    return arr


def max_misorientation_from_sym(sym_quats):
    sym_qs = _resolve_symmetry_quats(sym_quats)
    if sym_qs is None:
        return 180.0
    sym_qs = normalize_quat(sym_qs)
    max_angle = 0.0
    for i in range(len(sym_qs)):
        dots = np.abs(np.sum(sym_qs[i] * sym_qs, axis=-1))
        dots = np.clip(dots, -1.0, 1.0)
        max_angle = max(max_angle, float(np.max(2.0 * np.arccos(dots))))
    return float(np.degrees(max_angle))


def psnr_from_map(ref_map, test_map, max_val, eps=1e-10):
    ref_map = np.asarray(ref_map, dtype=np.float64)
    test_map = np.asarray(test_map, dtype=np.float64)
    mse = np.nanmean((ref_map - test_map) ** 2)
    if not np.isfinite(mse) or mse <= 0:
        return float("inf")
    return float(20.0 * np.log10(float(max_val) / np.sqrt(mse + eps)))


def ssim_from_map(ref_map, test_map, win_size=7, data_range=None):
    if not SKIMAGE_AVAILABLE or sk_ssim is None:
        return np.nan
    ref_map = np.asarray(ref_map, dtype=np.float64)
    test_map = np.asarray(test_map, dtype=np.float64)
    if data_range is None:
        data_range = float(
            np.nanmax([np.nanmax(ref_map), np.nanmax(test_map)])
            - np.nanmin([np.nanmin(ref_map), np.nanmin(test_map)])
        )
    if not np.isfinite(data_range) or data_range <= 0:
        return 1.0
    win_size = min(int(win_size), ref_map.shape[0], ref_map.shape[1])
    if win_size % 2 == 0:
        win_size -= 1
    win_size = max(win_size, 3)
    return float(
        sk_ssim(ref_map, test_map, data_range=float(data_range), win_size=win_size, channel_axis=None)
    )




def ssim_from_rgb_map(ref_rgb, test_rgb, win_size=7, data_range=1.0):
    if not SKIMAGE_AVAILABLE or sk_ssim is None:
        return np.nan
    ref_rgb = np.asarray(ref_rgb, dtype=np.float64)
    test_rgb = np.asarray(test_rgb, dtype=np.float64)
    if ref_rgb.ndim != 3 or ref_rgb.shape[-1] != 3:
        raise ValueError(f"Expected RGB map (H, W, 3), got {ref_rgb.shape}")
    if test_rgb.shape != ref_rgb.shape:
        raise ValueError(f"SSIM RGB shape mismatch: {test_rgb.shape} vs {ref_rgb.shape}")
    win_size = min(int(win_size), ref_rgb.shape[0], ref_rgb.shape[1])
    if win_size % 2 == 0:
        win_size -= 1
    win_size = max(win_size, 3)
    return float(
        sk_ssim(ref_rgb, test_rgb, data_range=float(data_range), win_size=win_size, channel_axis=-1)
    )



def ssim_on_ipf(pred_q, gt_q, sym_ops=None, win_size=7, ref_dir="Z"):
    sym_class = sym_ops if sym_ops is not None else sym
    ipf_pred = render_ipf_rgb(pred_q, sym_class, ref_dir=ref_dir)
    ipf_gt = render_ipf_rgb(gt_q, sym_class, ref_dir=ref_dir)
    return ssim_from_rgb_map(ipf_gt, ipf_pred, win_size=win_size, data_range=1.0)

def psnr_on_misorientation(pred_q, gt_q, sym_quats=None):
    mis = misorientation_map(pred_q, gt_q, sym_quats=sym_quats, degrees=True)
    return psnr_from_map(np.zeros_like(mis), mis, max_val=max_misorientation_from_sym(sym_quats))


def ssim_on_misorientation(pred_q, gt_q, sym_quats=None, win_size=7):
    mis = misorientation_map(pred_q, gt_q, sym_quats=sym_quats, degrees=True)
    return ssim_from_map(
        np.zeros_like(mis),
        mis,
        win_size=win_size,
        data_range=max_misorientation_from_sym(sym_quats),
    )


In [4]:
# ── Section 3: Load models + Open718 evaluation sample ────────────────────────
# The Open718 paper-side file is HR-only. For a 4×4 SR evaluation we use the
# top-left HR region divisible by 4 and derive LR by stride-4 sampling.
OPEN718_HR_FILE = Path("/data/home/umang/Materials/Reynolds-QSR_paper/Paper/EBSD_SR_Nature_v3/Open_718_Test_hr_x_block_0.npy")
OPEN718_IPFZ_REF_PNG = Path("/data/home/umang/Materials/Reynolds-QSR_paper/Paper/EBSD_SR_Nature_v3/Open_718_Test_hr_x_block_0_ipf_z.png")
sample_id = "Open_718_Test_hr_x_block_0"
sample_scale = (4, 4)
FIG_ROOT = repo_root / "analysis" / "out" / "patch_sample_comparison_all_methods_4x4" / sample_id
FULL_MAP_FIG_DIR = FIG_ROOT / "full_map"
FULL_MAP_PANEL_DIR = FULL_MAP_FIG_DIR / "ipf_xyz_panels"
BOUNDARY_FIG_DIR = FIG_ROOT / "boundary"
BOUNDARY_MASK_DIR = BOUNDARY_FIG_DIR / "masks"
PATCH_FIG_DIR = FIG_ROOT / "patch"
METRICS_FIG_DIR = FIG_ROOT / "metrics"
for _dir in (FULL_MAP_FIG_DIR, FULL_MAP_PANEL_DIR, BOUNDARY_FIG_DIR, BOUNDARY_MASK_DIR, PATCH_FIG_DIR, METRICS_FIG_DIR):
    _dir.mkdir(parents=True, exist_ok=True)
assert OPEN718_HR_FILE.exists(), f"Missing Open718 HR file: {OPEN718_HR_FILE}"

hr_raw_file = np.load(OPEN718_HR_FILE).astype(np.float32)
hr_passive_raw, RAW_SCALAR_LAYOUT, RAW_COMPONENT_ABS_MEAN = raw_to_passive_wxyz(hr_raw_file, scalar_layout="auto")
RAW_QUATERNION_CONVENTION = "passive"

hr_raw = format_quaternions(
    hr_passive_raw,
    normalize=True,
    hemisphere=True,
    reduce_fz=True,
    sym=sym,
    to_quat_first=False,
).astype(np.float32, copy=False)

aligned_h = (hr_raw.shape[0] // sample_scale[0]) * sample_scale[0]
aligned_w = (hr_raw.shape[1] // sample_scale[1]) * sample_scale[1]
hr = hr_raw[:aligned_h, :aligned_w].copy()
lr = hr[::sample_scale[0], ::sample_scale[1]].copy()

LR_FILE = None  # derived from HR, not a separate file on disk
HR_FILE = OPEN718_HR_FILE

# Representative patch: a 70×100 crop placed inside the large blue IPF-Z grain.
# The crop was selected from the HR map by first building a 5° crystallographic
# boundary mask and then choosing an interior rectangle in the largest blue
# connected grain.  The validation below checks that the patch is entirely
# inside one HR grain and contains no 5° HR boundary pixels.
PATCH_ROWS = slice(143, 213)
PATCH_COLS = slice(111, 211)
hr_patch = hr[PATCH_ROWS, PATCH_COLS]

GRAIN_BOUNDARY_THRESHOLD_DEG = 5.0
GRAIN_LABEL_CONNECTIVITY = 8
HR_GRAIN_BOUNDARY_MASK = compute_boundary_mask(
    hr,
    sym_ops=sym,
    threshold_deg=GRAIN_BOUNDARY_THRESHOLD_DEG,
    connectivity=4,
)
HR_GRAIN_LABELS, HR_GRAIN_COUNT = ndi_label(
    ~HR_GRAIN_BOUNDARY_MASK,
    structure=np.ones((3, 3), dtype=bool),
)
hr_patch_grain_labels = HR_GRAIN_LABELS[PATCH_ROWS, PATCH_COLS]
PATCH_CENTER = ((PATCH_ROWS.start + PATCH_ROWS.stop) // 2, (PATCH_COLS.start + PATCH_COLS.stop) // 2)
PATCH_HR_GRAIN_LABEL = int(HR_GRAIN_LABELS[PATCH_CENTER])
PATCH_HR_GRAIN_FRACTION = float(np.mean(hr_patch_grain_labels == PATCH_HR_GRAIN_LABEL))
PATCH_HR_BOUNDARY_FRACTION = float(np.mean(HR_GRAIN_BOUNDARY_MASK[PATCH_ROWS, PATCH_COLS]))
PATCH_UNIQUE_HR_GRAIN_LABELS = sorted(int(x) for x in np.unique(hr_patch_grain_labels))

# Guardrail against the common convention bug: the sidecar IPF-Z PNG matches
# passive scalar-first `[w,x,y,z]`, not active-conjugated quaternions.
IPFZ_REFERENCE_VALIDATION = None
if OPEN718_IPFZ_REF_PNG.exists():
    ipfz_ref_crop_u8, ipfz_ref_crop_box, ipfz_ref_png_shape = extract_reference_ipfz_map_crop(OPEN718_IPFZ_REF_PNG)
    ipfz_rendered_u8 = rgb01_to_uint8(render_ipf_rgb(hr_passive_raw, sym, ref_dir="Z"))
    nearest_resample = getattr(Image, "Resampling", Image).NEAREST
    ipfz_rendered_resized_u8 = np.asarray(
        Image.fromarray(ipfz_rendered_u8).resize(
            (ipfz_ref_crop_u8.shape[1], ipfz_ref_crop_u8.shape[0]),
            resample=nearest_resample,
        )
    )
    ipfz_abs_diff = np.abs(ipfz_rendered_resized_u8.astype(np.int16) - ipfz_ref_crop_u8.astype(np.int16))
    IPFZ_REFERENCE_VALIDATION = {
        "reference_png": str(OPEN718_IPFZ_REF_PNG),
        "render_layout": "passive_wxyz",
        "crop_box_xyxy": tuple(int(x) for x in ipfz_ref_crop_box),
        "png_shape": tuple(int(x) for x in ipfz_ref_png_shape),
        "mae_rgb_uint8": float(ipfz_abs_diff.mean()),
        "median_rgb_uint8": float(np.median(ipfz_abs_diff)),
        "p95_rgb_uint8": float(np.percentile(ipfz_abs_diff, 95)),
    }

print(f"Sample id: {sample_id}")
print(f"HR source: {HR_FILE}")
print("LR source: derived from aligned HR by stride-4 sampling")
print("HR raw file shape:", hr_raw_file.shape)
print("raw component |mean|:", RAW_COMPONENT_ABS_MEAN)
print("detected raw scalar layout:", RAW_SCALAR_LAYOUT, "([x,y,z,w] if xyzw)")
print("raw quaternion convention:", RAW_QUATERNION_CONVENTION)
print("HR formatted passive wxyz shape:", hr_raw.shape)
print("HR aligned shape:", hr.shape)
print("LR derived shape:", lr.shape)
print("Patch shape:", hr_patch.shape)
print("Patch rows/cols:", (PATCH_ROWS.start, PATCH_ROWS.stop), (PATCH_COLS.start, PATCH_COLS.stop))
print("Patch selection: large blue IPF-Z grain interior")
print("HR grain labels in patch:", PATCH_UNIQUE_HR_GRAIN_LABELS)
print("Patch center HR grain label:", PATCH_HR_GRAIN_LABEL)
print("Patch fraction inside center HR grain:", f"{PATCH_HR_GRAIN_FRACTION:.4f}")
print("Patch HR boundary-pixel fraction at 5°:", f"{PATCH_HR_BOUNDARY_FRACTION:.4f}")
print("Sample scale:", sample_scale)
print("Figure output root:", FIG_ROOT)
print("IPF-Z display quaternion layout: passive_wxyz")
print("IPF-Z reference validation:", IPFZ_REFERENCE_VALIDATION)


def _as_scale_tuple(scale_value):
    if isinstance(scale_value, (list, tuple)):
        return (int(scale_value[0]), int(scale_value[1]))
    scale_value = int(scale_value)
    return (scale_value, scale_value)


def _resolve_run_cfg_path(exp_dir):
    candidates = [
        exp_dir / "logs" / "inference_run_config.json",
        exp_dir / "logs" / "run_config.json",
    ]
    for path in candidates:
        if path.exists():
            return path
    # load_and_prepare_config writes the resolved config when given a save_path,
    # so a missing logs/run_config file should not be fatal for optional models.
    return exp_dir / "logs" / "inference_run_config.json"


def _resolve_checkpoint_path(cfg_local, exp_dir, checkpoint_name="best_model.pt"):
    checkpoints_dir = Path(getattr(cfg_local, "checkpoints_dir", exp_dir / "checkpoints"))
    if checkpoint_name is not None:
        return inf._resolve_checkpoint(cfg_local, exp_dir, checkpoint_name)

    candidates = sorted(checkpoints_dir.glob("*.pt"))
    if not candidates:
        raise FileNotFoundError(f"No checkpoints found under {checkpoints_dir}")

    def _checkpoint_rank(path):
        name = path.name
        if name == "last_checkpoint.pt":
            return (3, 0)
        if name.startswith("epoch_") and name.endswith(".pt"):
            epoch_text = name[len("epoch_"):-len(".pt")]
            if epoch_text.isdigit():
                return (2, int(epoch_text))
        if name == "best_model.pt":
            return (1, 0)
        return (0, path.stat().st_mtime_ns)

    return max(candidates, key=_checkpoint_rank)


def _load_experiment_model(exp_dir, *, override_scale=None, checkpoint_name="best_model.pt"):
    cfg_path = exp_dir / "config_new.json"
    run_cfg_path = _resolve_run_cfg_path(exp_dir)
    assert cfg_path.exists(), f"Config missing: {cfg_path}"
    cfg_local = load_and_prepare_config(cfg_path, run_cfg_path)
    if override_scale is not None:
        override_scale = _as_scale_tuple(override_scale)
        original_scale = _as_scale_tuple(getattr(cfg_local, "upsample_factor", getattr(cfg_local, "scale", override_scale)))
        if original_scale != override_scale:
            print(f"Overriding {exp_dir.name} scale from {original_scale} to {override_scale} for this sample.")
        cfg_local.scale = list(override_scale)
        cfg_local.upsample_factor = list(override_scale)
    checkpoint = _resolve_checkpoint_path(cfg_local, exp_dir, checkpoint_name)
    assert checkpoint.exists(), f"Checkpoint missing: {checkpoint}"
    model_local = inf._load_model_from_checkpoint(cfg_local, checkpoint, device)
    return cfg_local, model_local, checkpoint


# Paper-aligned learned 4×4 baselines from this repository.
# The paper calls the anchorless OCRP checkpoint simply "OCRP"; the longer
# variable name is kept so it remains obvious that this is the active-convention
# epoch-24 module checked in the dedicated proof/pass notebooks.
cfg = model = checkpoint = None
base_model_scale = None
model_standard_ocrp = None
model_macro_ocrp = None

new_anchorless_exp_dir = repo_root / "experiments" / "IN718" / "iso_embedding_4x4_ocrp_anchorless_4x1clone_01"
cfg_new_anchorless_ocrp = model_new_anchorless_ocrp = checkpoint_new_anchorless_ocrp = None
try:
    cfg_new_anchorless_ocrp, model_new_anchorless_ocrp, checkpoint_new_anchorless_ocrp = _load_experiment_model(
        new_anchorless_exp_dir,
        override_scale=sample_scale,
        checkpoint_name="epoch_0024.pt",
    )
    new_anchorless_scale = _as_scale_tuple(
        getattr(cfg_new_anchorless_ocrp, "upsample_factor", getattr(cfg_new_anchorless_ocrp, "scale", sample_scale))
    )
    print("OCRP anchorless epoch-24 loaded:", type(model_new_anchorless_ocrp).__name__, "from", checkpoint_new_anchorless_ocrp)
    print("OCRP native scale:", new_anchorless_scale)
except Exception as exc:
    print(f"Skipping OCRP anchorless epoch-24: {type(exc).__name__}: {exc}")

PAPER_4X4_JANGID_BASELINES = OrderedDict(
    [
        ("EDSR", repo_root / "experiments" / "IN718" / "edsr_4x4_01"),
        ("QEDSR", repo_root / "experiments" / "IN718" / "qedsr_4x4_01"),
        ("Q-RBSA-adapted", repo_root / "experiments" / "IN718" / "qrbsa_4x4_300ep_01"),
        ("HAN", repo_root / "experiments" / "IN718" / "han_4x4_300ep_01"),
        ("RCAN", repo_root / "experiments" / "IN718" / "rcan_4x4_300ep_01"),
        ("SAN", repo_root / "experiments" / "IN718" / "san_4x4_300ep_01"),
    ]
)


def _resolve_plain_checkpoint(exp_dir: Path, checkpoint_name: str = "best_model.pt") -> Path:
    checkpoint = Path(checkpoint_name)
    if checkpoint.is_absolute() or checkpoint.exists():
        return checkpoint
    return exp_dir / "checkpoints" / checkpoint_name


def _load_jangid_baseline(exp_dir: Path, checkpoint_name: str = "best_model.pt"):
    cfg_path = exp_dir / "config.json"
    checkpoint = _resolve_plain_checkpoint(exp_dir, checkpoint_name)
    if not cfg_path.exists():
        raise FileNotFoundError(f"Missing config: {cfg_path}")
    if not checkpoint.exists():
        raise FileNotFoundError(f"Missing checkpoint: {checkpoint}")
    cfg_local = json.loads(cfg_path.read_text())
    model_local = build_jangid_model(cfg_local).to(device)
    ckpt = torch.load(checkpoint, map_location=device, weights_only=False)
    state = ckpt.get("model_state_dict", ckpt)
    model_local.load_state_dict(state, strict=True)
    model_local.eval()
    return cfg_local, model_local, checkpoint


paper_jangid_models = OrderedDict()
for method_name, exp_dir in PAPER_4X4_JANGID_BASELINES.items():
    try:
        cfg_local, model_local, checkpoint_local = _load_jangid_baseline(exp_dir)
        paper_jangid_models[method_name] = {
            "cfg": cfg_local,
            "model": model_local,
            "checkpoint": checkpoint_local,
            "exp_dir": exp_dir,
            "model_type": cfg_local.get("model_type"),
            "expected_trainable_params": cfg_local.get("expected_trainable_params"),
        }
        print(f"{method_name} loaded:", cfg_local.get("model_type"), "from", checkpoint_local)
    except Exception as exc:
        print(f"Skipping {method_name}: {type(exc).__name__}: {exc}")

atindama_exp_dir = repo_root / "experiments" / "IN718" / "atindama_inpainting_4x4_01"
cfg_atindama = model_atindama = checkpoint_atindama = None
try:
    atindama_cfg_path = atindama_exp_dir / "config.json"
    checkpoint_atindama = _resolve_plain_checkpoint(atindama_exp_dir, "best_model.pt")
    cfg_atindama = json.loads(atindama_cfg_path.read_text())
    model_atindama = load_authors_model(device)
    ckpt = torch.load(checkpoint_atindama, map_location=device, weights_only=False)
    model_atindama.load_state_dict(ckpt.get("model_state_dict", ckpt), strict=True)
    model_atindama.eval()
    print("Atindama inpainting loaded from", checkpoint_atindama)
except Exception as exc:
    print(f"Skipping Atindama inpainting: {type(exc).__name__}: {exc}")

PAPER_BASELINE_LOAD_STATUS = {
    "OCRP": model_new_anchorless_ocrp is not None,
    **{name: name in paper_jangid_models for name in PAPER_4X4_JANGID_BASELINES},
    "Atindama inpainting": model_atindama is not None,
}

print("LR shape:", lr.shape)
print("HR shape:", hr.shape)
print("Patch shape:", hr_patch.shape)
print("Sample scale:", sample_scale)
print("Paper 4×4 learned baseline load status:")
for name, ok in PAPER_BASELINE_LOAD_STATUS.items():
    print(f"  {name}: {'available' if ok else 'missing/skipped'}")


Sample id: Open_718_Test_hr_x_block_0
HR source: /data/home/umang/Materials/Reynolds-QSR_paper/Paper/EBSD_SR_Nature_v3/Open_718_Test_hr_x_block_0.npy
LR source: derived from aligned HR by stride-4 sampling
HR raw file shape: (301, 390, 4)
raw component |mean|: [0.18778956 0.18947135 0.14824016 0.93412596]
detected raw scalar layout: xyzw ([x,y,z,w] if xyzw)
raw quaternion convention: passive
HR formatted passive wxyz shape: (301, 390, 4)
HR aligned shape: (300, 388, 4)
LR derived shape: (75, 97, 4)
Patch shape: (70, 100, 4)
Patch rows/cols: (143, 213) (111, 211)
Patch selection: large blue IPF-Z grain interior
HR grain labels in patch: [79]
Patch center HR grain label: 79
Patch fraction inside center HR grain: 1.0000
Patch HR boundary-pixel fraction at 5°: 0.0000
Sample scale: (4, 4)
Figure output root: /data/home/umang/Materials/Reynolds-QSR_paper/analysis/out/patch_sample_comparison_all_methods_4x4/Open_718_Test_hr_x_block_0
IPF-Z display quaternion layout: passive_wxyz
IPF-Z referen

In [5]:
# ── Section 3b: Paper-baseline availability sanity check ─────────────────────
print(f"LR shape: {lr.shape}")
print(f"HR shape: {hr.shape}")
print(f"Sample scale: {sample_scale}")
print(f"Expected HR shape: {(lr.shape[0] * sample_scale[0], lr.shape[1] * sample_scale[1], 4)}")
print("Paper 4×4 learned baselines loaded from experiments/IN718:")
for name, ok in PAPER_BASELINE_LOAD_STATUS.items():
    print(f"  {name:22s}: {'yes' if ok else 'no'}")


LR shape: (75, 97, 4)
HR shape: (300, 388, 4)
Sample scale: (4, 4)
Expected HR shape: (300, 388, 4)
Paper 4×4 learned baselines loaded from experiments/IN718:
  OCRP                  : yes
  EDSR                  : yes
  QEDSR                 : yes
  Q-RBSA-adapted        : yes
  HAN                   : yes
  RCAN                  : yes
  SAN                   : yes
  Atindama inpainting   : yes


In [6]:
# ── Section 4: Model / bicubic / NN methods ─────────────────────────────────
def _hwc4(arr):
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim != 3:
        raise ValueError(f"Expected rank-3 array, got {arr.shape}")
    if arr.shape[-1] == 4:
        return arr
    if arr.shape[0] == 4:
        return np.moveaxis(arr, 0, -1)
    raise ValueError(f"No axis of size 4 found in {arr.shape}")


model_bicubic_finterp = QuaternionBicubicFInterpolateSR(
    upsample_factor=sample_scale,
    align_corners=False,
    normalize_output=True,
    canonicalize_output=False,
    device=device,
).eval()


def _decode_features_with_internal_grad(sr_model, feat_hr):
    feat_hr = feat_hr.detach()
    with torch.enable_grad():
        return sr_model.decode(feat_hr)


def _forward_sr_lightweight(sr_model, lr_flat, lr_shape, *, normalize_input=True):
    lr_flat = lr_flat.to(device=sr_model.device, dtype=torch.float32)
    if normalize_input:
        lr_flat = F.normalize(lr_flat, p=2, dim=-1, eps=1e-12)

    with torch.no_grad():
        if hasattr(sr_model, "ocrp"):
            feat_lr = sr_model.encode(lr_flat)
            lr_for_ocrp = (
                lr_flat.view(feat_lr.shape[0], feat_lr.shape[1], 4)
                if feat_lr.dim() == 3
                else lr_flat
            )
            feat_hr, _ = sr_model._forward_sr_features(
                lr_quats=lr_for_ocrp,
                feat_lr=feat_lr,
                lr_shape=lr_shape,
                return_aux=False,
            )
        else:
            feat_lr = sr_model.encode_a1(lr_flat)
            feat_hr, _ = sr_model._forward_sr_features(feat_lr, lr_shape)
    return _decode_features_with_internal_grad(sr_model, feat_hr)


def upsample_model_sr(lr_q, out_hw):
    lr_q = _hwc4(lr_q)
    lr_t = torch.from_numpy(lr_q).to(device=device, dtype=torch.float32)
    lr_flat, lr_shape = inf._flatten_quat_chw(lr_t)
    sr_flat = _forward_sr_lightweight(model, lr_flat, lr_shape, normalize_input=True)
    return sr_flat.detach().cpu().numpy().reshape(int(out_hw[0]), int(out_hw[1]), 4)


def upsample_new_anchorless_ocrp_sr(lr_q, out_hw):
    if model_new_anchorless_ocrp is None:
        raise RuntimeError("New OCRP anchorless epoch-24 model not available")
    return _upsample_sr_tiled(
        model_new_anchorless_ocrp,
        lr_q,
        out_hw,
        tile_lr_shape=(16, 64),
        context_lr=(16, 20),
    )


def upsample_standard_ocrp_sr(lr_q, out_hw):
    if model_standard_ocrp is None:
        raise RuntimeError("Standard OCRP model not available")
    return _upsample_sr_tiled(
        model_standard_ocrp,
        lr_q,
        out_hw,
        tile_lr_shape=(16, 64),
        context_lr=(16, 20),
    )


def upsample_macro_ocrp_sr(lr_q, out_hw):
    if model_macro_ocrp is None:
        raise RuntimeError("Macro OCRP model not available")
    return _upsample_sr_tiled(
        model_macro_ocrp,
        lr_q,
        out_hw,
        tile_lr_shape=(16, 64),
        context_lr=(14, 24),
    )


def _upsample_sr_tiled(
    sr_model,
    lr_q,
    out_hw,
    *,
    tile_lr_shape=(16, 64),
    context_lr=(16, 20),
):
    lr_q = _hwc4(lr_q)
    h_lr, w_lr = lr_q.shape[:2]
    h_out, w_out = int(out_hw[0]), int(out_hw[1])
    scale_y = h_out // h_lr
    scale_x = w_out // w_lr
    if (h_lr * scale_y, w_lr * scale_x) != (h_out, w_out):
        raise ValueError(
            f"Output shape {(h_out, w_out)} is not an integer scale of LR shape {(h_lr, w_lr)}"
        )

    tile_h, tile_w = int(tile_lr_shape[0]), int(tile_lr_shape[1])
    ctx_y, ctx_x = int(context_lr[0]), int(context_lr[1])
    sr_full = np.zeros((h_out, w_out, 4), dtype=np.float32)

    for y0 in range(0, h_lr, tile_h):
        y1 = min(y0 + tile_h, h_lr)
        for x0 in range(0, w_lr, tile_w):
            x1 = min(x0 + tile_w, w_lr)
            crop_y0 = max(0, y0 - ctx_y)
            crop_y1 = min(h_lr, y1 + ctx_y)
            crop_x0 = max(0, x0 - ctx_x)
            crop_x1 = min(w_lr, x1 + ctx_x)

            lr_crop = lr_q[crop_y0:crop_y1, crop_x0:crop_x1]
            lr_t = torch.from_numpy(lr_crop).to(device=device, dtype=torch.float32)
            lr_flat, lr_shape = inf._flatten_quat_chw(lr_t)
            sr_crop = _forward_sr_lightweight(
                sr_model,
                lr_flat,
                lr_shape,
                normalize_input=True,
            )
            sr_crop = sr_crop.detach().cpu().numpy().reshape(
                (crop_y1 - crop_y0) * scale_y,
                (crop_x1 - crop_x0) * scale_x,
                4,
            )

            core_y0 = (y0 - crop_y0) * scale_y
            core_y1 = core_y0 + (y1 - y0) * scale_y
            core_x0 = (x0 - crop_x0) * scale_x
            core_x1 = core_x0 + (x1 - x0) * scale_x
            sr_full[y0 * scale_y:y1 * scale_y, x0 * scale_x:x1 * scale_x] = sr_crop[
                core_y0:core_y1,
                core_x0:core_x1,
            ]

            if device.type == "cuda":
                torch.cuda.empty_cache()

    return sr_full



def _normalize_hwc_quat_np(q: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    q = np.asarray(q, dtype=np.float32)
    q = q / np.maximum(np.linalg.norm(q, axis=-1, keepdims=True), eps)
    return np.where(q[..., :1] < 0.0, -q, q).astype(np.float32, copy=False)


def _canonicalize_passive_output(q: np.ndarray) -> np.ndarray:
    """Normalize and FZ-reduce passive scalar-first output for fair IPF/metric use."""
    q = _normalize_hwc_quat_np(q)
    proper = sym.proper_subgroup if hasattr(sym, "proper_subgroup") else sym
    try:
        q = reduce_to_fz_min_angle(
            q,
            sym=proper,
            normalize=True,
            hemisphere=True,
            return_op_map=False,
        ).astype(np.float32, copy=False)
    except Exception:
        q = _normalize_hwc_quat_np(q)
    return q


def _fallback_invalid_pixels(sr_np: np.ndarray, lr_q: np.ndarray, out_hw) -> np.ndarray:
    finite = np.isfinite(sr_np).all(axis=-1, keepdims=True)
    if bool(finite.all()):
        return sr_np
    fallback = upsample_nn(lr_q, out_hw)
    print(f"  replaced {int((~finite[..., 0]).sum())} non-finite pixels with nearest-neighbor LR fallback")
    return np.where(finite, sr_np, fallback).astype(np.float32, copy=False)


def _upsample_jangid_baseline(method_name: str, lr_q, out_hw):
    if method_name not in paper_jangid_models:
        raise RuntimeError(f"{method_name} checkpoint not loaded")
    baseline = paper_jangid_models[method_name]
    baseline_model = baseline["model"]
    lr_q = _hwc4(lr_q)
    lr_passive_chw = jangid_to_chw(torch.from_numpy(lr_q[None]).to(device=device, dtype=torch.float32))
    with torch.no_grad():
        sr_active_chw = forward_baseline_active_from_passive(baseline_model, lr_passive_chw)
        sr_passive_chw = jangid_conjugate_scalar_first_chw(sr_active_chw)
    sr_np = jangid_to_hwc_numpy(sr_passive_chw[0]).astype(np.float32, copy=False)
    sr_np = _fallback_invalid_pixels(sr_np, lr_q, out_hw)
    expected = (int(out_hw[0]), int(out_hw[1]))
    if sr_np.shape[:2] != expected:
        raise ValueError(f"{method_name} output shape {sr_np.shape[:2]} != expected {expected}")
    return _canonicalize_passive_output(sr_np)


def upsample_edsr_sr(lr_q, out_hw):
    return _upsample_jangid_baseline("EDSR", lr_q, out_hw)


def upsample_qedsr_sr(lr_q, out_hw):
    return _upsample_jangid_baseline("QEDSR", lr_q, out_hw)


def upsample_qrbsa_adapted_sr(lr_q, out_hw):
    return _upsample_jangid_baseline("Q-RBSA-adapted", lr_q, out_hw)


def upsample_han_sr(lr_q, out_hw):
    return _upsample_jangid_baseline("HAN", lr_q, out_hw)


def upsample_rcan_sr(lr_q, out_hw):
    return _upsample_jangid_baseline("RCAN", lr_q, out_hw)


def upsample_san_sr(lr_q, out_hw):
    return _upsample_jangid_baseline("SAN", lr_q, out_hw)


def upsample_atindama_inpainting_sr(lr_q, out_hw):
    if model_atindama is None:
        raise RuntimeError("Atindama inpainting checkpoint not loaded")
    h_out, w_out = int(out_hw[0]), int(out_hw[1])
    if (h_out, w_out) != hr.shape[:2]:
        raise ValueError("Atindama wrapper expects the notebook's aligned HR field as the target grid")

    # Atindama is an inpainting baseline.  We build the same periodic known-pixel
    # mask used by inference/infer_atindama_inpainting.py: only stride-4 samples
    # are visible to the network; unknown HR pixels are zeroed before inference.
    target_euler = passive_quaternion_to_normalized_zxz(hr[:h_out, :w_out])
    known_mask_np = periodic_known_mask(h_out, w_out, sample_scale)
    target = torch.from_numpy(np.moveaxis(target_euler, -1, 0)[None]).to(device=device, dtype=torch.float32)
    known_mask = torch.from_numpy(known_mask_np[None]).to(device=device, dtype=torch.float32)

    # The authors' U-Net has seven stride-2 encoder stages, so dimensions need
    # to be compatible with 2^7 down/up sampling.  The Open718 field is 300×388,
    # so pad the network canvas to a safe multiple and mark all padded pixels as
    # unknown.  We crop back before
    # converting to quaternions.
    pad_multiple = 128
    pad_h = (pad_multiple - h_out % pad_multiple) % pad_multiple
    pad_w = (pad_multiple - w_out % pad_multiple) % pad_multiple
    target_padded = F.pad(target, (0, pad_w, 0, pad_h), mode="constant", value=0.0)
    known_mask_padded = F.pad(known_mask, (0, pad_w, 0, pad_h), mode="constant", value=0.0)

    with torch.no_grad():
        prediction, _ = model_atindama(target_padded * known_mask_padded, known_mask_padded)
        composite, invalid_counts = sanitize_prediction(prediction, target_padded, known_mask_padded, sample_scale)
    if int(invalid_counts.sum().item()) > 0:
        print("  Atindama invalid predicted pixels replaced:", int(invalid_counts.sum().item()))
    euler_hwc = composite[0, :, :h_out, :w_out].permute(1, 2, 0).detach().cpu().numpy()
    sr_np = normalized_zxz_to_passive_quaternion(euler_hwc)
    return _canonicalize_passive_output(sr_np)

def upsample_bicubic(lr_q, out_hw):
    lr_q = _hwc4(lr_q)
    h_out, w_out = int(out_hw[0]), int(out_hw[1])
    sr = np.zeros((h_out, w_out, 4), dtype=np.float32)
    for i in range(4):
        sr[..., i] = cv2.resize(lr_q[..., i], (w_out, h_out), interpolation=cv2.INTER_CUBIC)
    norms = np.linalg.norm(sr, axis=-1, keepdims=True)
    norms[norms == 0] = 1.0
    return sr / norms


def upsample_bicubic_finterp(lr_q, out_hw):
    lr_q = _hwc4(lr_q)
    lr_t = torch.from_numpy(lr_q).to(device=device, dtype=torch.float32)
    lr_flat, lr_shape = inf._flatten_quat_chw(lr_t)
    with torch.no_grad():
        sr_flat = model_bicubic_finterp.forward_sr(
            lr_flat,
            lr_shape=lr_shape,
            normalize_input=False,
        )
    sr_np = sr_flat.detach().cpu().numpy().reshape(int(out_hw[0]), int(out_hw[1]), 4)
    if sr_np.shape[:2] != (int(out_hw[0]), int(out_hw[1])):
        raise ValueError(
            f"Torch bicubic SR shape mismatch: got {sr_np.shape[:2]}, expected {(int(out_hw[0]), int(out_hw[1]))}"
        )
    return sr_np.astype(np.float32, copy=False)


def upsample_iso_bicubic(lr_q, out_hw):
    lr_q = _hwc4(lr_q)
    h_lr, w_lr = lr_q.shape[:2]
    h_out, w_out = int(out_hw[0]), int(out_hw[1])

    lr_t = torch.from_numpy(lr_q.reshape(-1, 4)).to(device=device, dtype=torch.float32)
    with torch.no_grad():
        feat_lr = model.encode_a1(lr_t)
    c = feat_lr.shape[-1]
    feat_lr_hwc = feat_lr.cpu().numpy().reshape(h_lr, w_lr, c)

    feat_hr_hwc = np.zeros((h_out, w_out, c), dtype=np.float32)
    for i in range(c):
        feat_hr_hwc[..., i] = cv2.resize(
            feat_lr_hwc[..., i], (w_out, h_out), interpolation=cv2.INTER_CUBIC
        )

    feat_hr_flat = torch.from_numpy(feat_hr_hwc.reshape(h_out * w_out, c)).to(
        device=device, dtype=torch.float32
    )
    sr_t = _decode_features_with_internal_grad(model, feat_hr_flat)
    return sr_t.detach().cpu().numpy().reshape(h_out, w_out, 4)


def upsample_nn(lr_q, out_hw):
    lr_q = _hwc4(lr_q)
    h_out, w_out = int(out_hw[0]), int(out_hw[1])
    sr = np.zeros((h_out, w_out, 4), dtype=np.float32)
    for i in range(4):
        sr[..., i] = cv2.resize(lr_q[..., i], (w_out, h_out), interpolation=cv2.INTER_NEAREST)
    norms = np.linalg.norm(sr, axis=-1, keepdims=True)
    norms[norms == 0] = 1.0
    return sr / norms


def upsample_iso_nn(lr_q, out_hw):
    lr_q = _hwc4(lr_q)
    h_lr, w_lr = lr_q.shape[:2]
    h_out, w_out = int(out_hw[0]), int(out_hw[1])

    lr_t = torch.from_numpy(lr_q.reshape(-1, 4)).to(device=device, dtype=torch.float32)
    with torch.no_grad():
        feat_lr = model.encode_a1(lr_t)
    c = feat_lr.shape[-1]
    feat_lr_hwc = feat_lr.cpu().numpy().reshape(h_lr, w_lr, c)

    feat_hr_hwc = np.zeros((h_out, w_out, c), dtype=np.float32)
    for i in range(c):
        feat_hr_hwc[..., i] = cv2.resize(
            feat_lr_hwc[..., i], (w_out, h_out), interpolation=cv2.INTER_NEAREST
        )

    feat_hr_flat = torch.from_numpy(feat_hr_hwc.reshape(h_out * w_out, c)).to(
        device=device, dtype=torch.float32
    )
    sr_t = _decode_features_with_internal_grad(model, feat_hr_flat)
    return sr_t.detach().cpu().numpy().reshape(h_out, w_out, 4)


In [7]:
# ── Section 5: SLERP / Symm-SLERP methods ───────────────────────────────────
from slerp_final import (
    qnorm as torch_qnorm,
    slerp as torch_slerp,
    symmetrize_pair as torch_symmetrize_pair,
    make_fcc_symmetry_4x4,
)

interp_device = torch.device("cpu")
sym_ops_oh = make_fcc_symmetry_4x4(device=interp_device, dtype=torch.float32)


def upsample_slerp_quat_torch(lr_q, out_hw, *, sym_ops_4x4=None, device=interp_device):
    lr_q = _hwc4(lr_q)
    h_out, w_out = int(out_hw[0]), int(out_hw[1])

    x = torch.from_numpy(lr_q).permute(2, 0, 1).unsqueeze(0).to(device=device, dtype=torch.float32)
    x = torch_qnorm(x)
    B, C, H, W = x.shape

    sy = float(h_out) / float(H)
    sx = float(w_out) / float(W)

    iy = torch.arange(h_out, device=device, dtype=x.dtype)
    ix = torch.arange(w_out, device=device, dtype=x.dtype)
    y = (iy + 0.5) / sy - 0.5
    xcoord = (ix + 0.5) / sx - 0.5

    y0 = torch.floor(y).clamp(0, H - 1).long()
    x0 = torch.floor(xcoord).clamp(0, W - 1).long()
    y1 = (y0 + 1).clamp(0, H - 1)
    x1 = (x0 + 1).clamp(0, W - 1)

    v = (y - y0.to(x.dtype)).clamp(0.0, 1.0)
    u = (xcoord - x0.to(x.dtype)).clamp(0.0, 1.0)
    v_grid, u_grid = torch.meshgrid(v, u, indexing="ij")

    y0g = y0.view(h_out, 1).expand(h_out, w_out)
    y1g = y1.view(h_out, 1).expand(h_out, w_out)
    x0g = x0.view(1, w_out).expand(h_out, w_out)
    x1g = x1.view(1, w_out).expand(h_out, w_out)

    q00 = x[:, :, y0g, x0g]
    q01 = x[:, :, y0g, x1g]
    q10 = x[:, :, y1g, x0g]
    q11 = x[:, :, y1g, x1g]

    def flat(q):
        return q.permute(0, 2, 3, 1).reshape(-1, 4)

    q00_f, q01_f = flat(q00), flat(q01)
    q10_f, q11_f = flat(q10), flat(q11)
    u_full = u_grid.unsqueeze(0).expand(B, -1, -1).reshape(-1)
    v_full = v_grid.unsqueeze(0).expand(B, -1, -1).reshape(-1)

    if sym_ops_4x4 is None:
        q0u = torch_slerp(q00_f, q01_f, u_full)
        q1u = torch_slerp(q10_f, q11_f, u_full)
        q_uv = torch_slerp(q0u, q1u, v_full)
    else:
        sym_ops_4x4 = sym_ops_4x4.to(device=device, dtype=x.dtype)
        q00m, q01m = torch_symmetrize_pair(q00_f, q01_f, sym_ops_4x4)
        q0u = torch_slerp(q00m, q01m, u_full)
        q10m, q11m = torch_symmetrize_pair(q10_f, q11_f, sym_ops_4x4)
        q1u = torch_slerp(q10m, q11m, u_full)
        q0u_m, q1u_m = torch_symmetrize_pair(q0u, q1u, sym_ops_4x4)
        q_uv = torch_slerp(q0u_m, q1u_m, v_full)

    q_uv = torch_qnorm(q_uv)
    return q_uv.view(B, h_out, w_out, 4)[0].detach().cpu().numpy().astype(np.float32)


def upsample_slerp(lr_q, out_hw):
    return upsample_slerp_quat_torch(lr_q, out_hw, sym_ops_4x4=None, device=interp_device)


def upsample_symm_slerp(lr_q, out_hw):
    return upsample_slerp_quat_torch(lr_q, out_hw, sym_ops_4x4=sym_ops_oh, device=interp_device)


In [8]:
# ── Section 6: Boundary-Aware Sym-SLERP method ──────────────────────────────
from slerp_final import (
    bilinear_slerp_sym,
    qnorm as _qnorm_t,
    slerp as _slerp_t,
    symmetrize_pair,
)
from segment_grains import (
    segment_grains_graph,
    cleanup_small_grains_cuda,
    compute_gb_mask as _compute_gb_mask_t,
)

_ba_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_ba_sym_ops_cpu = make_fcc_symmetry_4x4(device="cpu", dtype=torch.float32)


@torch.no_grad()
def _build_smooth_hr_labels_aniso(labels_lr, scale_y, scale_x=1, sdf_shift=0.7):
    dev = labels_lr.device
    H, W = labels_lr.shape
    H_hr, W_hr = H * scale_y, W * scale_x

    gb_lr = _compute_gb_mask_t(labels_lr)
    gb_hr = F.interpolate(gb_lr[None, None], size=(H_hr, W_hr), mode="nearest")[0, 0]

    kernel = torch.ones((1, 1, 3, 3), device=dev) / 9.0
    dist = gb_hr.clone()
    for _ in range(12):
        dist = F.conv2d(dist[None, None], kernel, padding=1)[0, 0]

    sigma = 2.0
    ks = int(6 * sigma + 1) | 1
    half = ks // 2
    coords = torch.arange(ks, device=dev, dtype=torch.float32) - half
    g1 = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g1 = g1 / g1.sum()
    g2 = torch.outer(g1, g1)[None, None]
    sdf_hr = F.conv2d(dist[None, None], g2, padding=half)[0, 0]
    sdf_hr = sdf_hr / (sdf_hr.max() + 1e-8)

    dy = torch.zeros_like(sdf_hr)
    dx = torch.zeros_like(sdf_hr)
    dy[1:-1, :] = 0.5 * (sdf_hr[2:, :] - sdf_hr[:-2, :])
    dy[0, :] = sdf_hr[1, :] - sdf_hr[0, :]
    dy[-1, :] = sdf_hr[-1, :] - sdf_hr[-2, :]
    dx[:, 1:-1] = 0.5 * (sdf_hr[:, 2:] - sdf_hr[:, :-2])
    dx[:, 0] = sdf_hr[:, 1] - sdf_hr[:, 0]
    dx[:, -1] = sdf_hr[:, -1] - sdf_hr[:, -2]
    norm = torch.sqrt(dx * dx + dy * dy + 1e-12)
    nx, ny = dx / norm, dy / norm

    idx_y = torch.arange(H_hr, device=dev, dtype=torch.float32)
    idx_x = torch.arange(W_hr, device=dev, dtype=torch.float32)
    y_base = (idx_y + 0.5) / scale_y - 0.5
    x_base = (idx_x + 0.5) / scale_x - 0.5
    y_grid, x_grid = torch.meshgrid(y_base, x_base, indexing="ij")

    y_nn = torch.round(y_grid - sdf_shift * ny).clamp(0, H - 1).long()
    x_nn = torch.round(x_grid - sdf_shift * nx).clamp(0, W - 1).long()
    return labels_lr[y_nn, x_nn].long()


@torch.no_grad()
def _analyze_boundary_cases_aniso(labels_lr, labels_hr, scale_y, scale_x=1):
    dev = labels_lr.device
    H_lr, W_lr = labels_lr.shape
    H_hr, W_hr = labels_hr.shape

    yy = torch.arange(H_hr, device=dev, dtype=torch.float32)
    xx = torch.arange(W_hr, device=dev, dtype=torch.float32)
    y_hr_g, x_hr_g = torch.meshgrid(yy, xx, indexing="ij")

    y_lr_f = (y_hr_g + 0.5) / scale_y - 0.5
    x_lr_f = (x_hr_g + 0.5) / scale_x - 0.5

    y0 = torch.floor(y_lr_f).long().clamp(0, H_lr - 1)
    x0 = torch.floor(x_lr_f).long().clamp(0, W_lr - 1)
    y1 = (y0 + 1).clamp(0, H_lr - 1)
    x1 = (x0 + 1).clamp(0, W_lr - 1)

    ty = (y_lr_f - y0.float()).clamp(0.0, 1.0)
    tx = (x_lr_f - x0.float()).clamp(0.0, 1.0)
    w_nb = torch.stack(
        [(1 - tx) * (1 - ty), tx * (1 - ty), (1 - tx) * ty, tx * ty], dim=-1
    )
    g_nb = torch.stack(
        [labels_lr[y0, x0], labels_lr[y0, x1], labels_lr[y1, x0], labels_lr[y1, x1]], dim=-1
    )
    same = g_nb == labels_hr.unsqueeze(-1)

    N = H_hr * W_hr
    sf = same.reshape(N, 4)
    wf = w_nb.reshape(N, 4)
    nv = sf.sum(dim=1)
    cf = torch.zeros(N, dtype=torch.uint8, device=dev)
    cf[nv == 4] = 3
    cf[(nv >= 2) & (nv < 4)] = 2
    cf[nv == 1] = 1
    return sf, wf, cf.view(H_hr, W_hr)


@torch.no_grad()
def _ba_slerp_core(q_lr_t, labels_lr, labels_hr, sym_ops_4x4,
                   scale_y, scale_x, same_mask_flat, w_flat, case_map):
    dev = q_lr_t.device
    dtype = q_lr_t.dtype
    _, _, H_lr, W_lr = q_lr_t.shape
    H_hr, W_hr = labels_hr.shape
    N = H_hr * W_hr

    yy = torch.arange(H_hr, device=dev, dtype=torch.float32)
    xx = torch.arange(W_hr, device=dev, dtype=torch.float32)
    y_g, x_g = torch.meshgrid(yy, xx, indexing="ij")
    y_lr_f = (y_g + 0.5) / scale_y - 0.5
    x_lr_f = (x_g + 0.5) / scale_x - 0.5

    y0 = torch.floor(y_lr_f).long().clamp(0, H_lr - 1)
    x0 = torch.floor(x_lr_f).long().clamp(0, W_lr - 1)
    y1 = (y0 + 1).clamp(0, H_lr - 1)
    x1 = (x0 + 1).clamp(0, W_lr - 1)
    u_f = (x_lr_f - x0.float()).clamp(0.0, 1.0).reshape(-1)
    v_f = (y_lr_f - y0.float()).clamp(0.0, 1.0).reshape(-1)

    q_hw = q_lr_t.squeeze(0).permute(1, 2, 0)
    q00_f = q_hw[y0, x0].reshape(N, 4)
    q01_f = q_hw[y0, x1].reshape(N, 4)
    q10_f = q_hw[y1, x0].reshape(N, 4)
    q11_f = q_hw[y1, x1].reshape(N, 4)
    q_nb = torch.stack([q00_f, q01_f, q10_f, q11_f], dim=1)

    cf = case_map.reshape(-1)
    gid_f = labels_hr.reshape(-1)
    y_est_f = y_lr_f.reshape(-1)
    x_est_f = x_lr_f.reshape(-1)
    q_out = torch.zeros((N, 4), dtype=dtype, device=dev)

    idx3 = (cf == 3).nonzero(as_tuple=False).squeeze(1)
    if idx3.numel() > 0:
        q_out[idx3] = bilinear_slerp_sym(
            q00_f[idx3], q01_f[idx3], q10_f[idx3], q11_f[idx3],
            u_f[idx3], v_f[idx3], sym_ops_4x4,
        )

    idx1 = (cf == 1).nonzero(as_tuple=False).squeeze(1)
    if idx1.numel() > 0:
        pos1 = same_mask_flat[idx1].long().argmax(dim=1)
        q_out[idx1] = q_nb[idx1, pos1]

    idx2 = (cf == 2).nonzero(as_tuple=False).squeeze(1)
    if idx2.numel() > 0:
        q_nb2 = q_nb[idx2]
        w2 = w_flat[idx2]
        same2 = same_mask_flat[idx2]
        for i in range(idx2.numel()):
            valid = same2[i].nonzero(as_tuple=False).squeeze(1)
            if valid.numel() == 0:
                continue
            qv = q_nb2[i, valid]
            wv = w2[i, valid] / (w2[i, valid].sum() + 1e-12)
            q_ref = qv[0:1]
            total_w = wv[0].clone()
            for k in range(1, valid.numel()):
                alpha = wv[k] / (total_w + wv[k] + 1e-12)
                q_ref, q_nxt = symmetrize_pair(q_ref, qv[k:k + 1], sym_ops_4x4)
                q_ref = _slerp_t(
                    q_ref, q_nxt, torch.tensor([float(alpha)], device=dev, dtype=dtype)
                )
                total_w = total_w + wv[k]
            q_out[idx2[i]] = q_ref[0]

    idx0 = (cf == 0).nonzero(as_tuple=False).squeeze(1)
    if idx0.numel() > 0:
        lab_cpu = labels_lr.cpu()
        q_hw_cpu = q_hw.cpu()
        for j in idx0.tolist():
            gid = int(gid_f[j].item())
            ye = int(round(float(y_est_f[j].item())))
            xe = int(round(float(x_est_f[j].item())))
            best_q = None
            for r in range(1, 4):
                if best_q is not None:
                    break
                for dy in range(-r, r + 1):
                    for dx in range(-r, r + 1):
                        yy_ = min(max(ye + dy, 0), H_lr - 1)
                        xx_ = min(max(xe + dx, 0), W_lr - 1)
                        if int(lab_cpu[yy_, xx_].item()) == gid:
                            best_q = q_hw_cpu[yy_, xx_]
                            break
                    if best_q is not None:
                        break
            if best_q is None:
                best_q = q_hw_cpu[min(max(ye, 0), H_lr - 1), min(max(xe, 0), W_lr - 1)]
            q_out[j] = _qnorm_t(best_q.to(dev, dtype).unsqueeze(0))[0]

    q_hr = _qnorm_t(q_out.view(H_hr, W_hr, 4).permute(2, 0, 1).unsqueeze(0))
    return q_hr


def upsample_ba_sym_slerp(lr_q, out_hw):
    lr_q = _hwc4(lr_q)
    q_lr_cpu = _qnorm_t(torch.from_numpy(lr_q).permute(2, 0, 1).unsqueeze(0).float())
    labels_np, _ = segment_grains_graph(q_lr_cpu, _ba_sym_ops_cpu, thr_deg=3.0)
    labels_lr = torch.from_numpy(labels_np).long()
    if torch.cuda.is_available():
        labels_lr = cleanup_small_grains_cuda(
            labels_lr.to(_ba_device), q_lr_cpu.to(_ba_device), _ba_sym_ops_cpu.to(_ba_device),
            min_pixels=3, max_iter=1,
        ).cpu()
    labels_hr = _build_smooth_hr_labels_aniso(labels_lr, 4, 4, sdf_shift=0.7)
    same_mask_flat, w_flat, case_map = _analyze_boundary_cases_aniso(labels_lr, labels_hr, 4, 4)
    q_hr = _ba_slerp_core(
        q_lr_cpu.to(_ba_device),
        labels_lr.to(_ba_device),
        labels_hr.to(_ba_device),
        _ba_sym_ops_cpu.to(_ba_device),
        4,
        4,
        same_mask_flat.to(_ba_device),
        w_flat.to(_ba_device),
        case_map.to(_ba_device),
    )
    return q_hr.squeeze(0).permute(1, 2, 0).cpu().numpy().astype(np.float32)


In [9]:
# ── Section 7: Methods registry + single-sample evaluation harness ──────────
METHODS = OrderedDict()

# Paper learned baselines first.  These labels intentionally match the paper
# summary tables, with OCRP referring to the anchorless epoch-24 checkpoint.
if model_new_anchorless_ocrp is not None:
    METHODS["OCRP"] = upsample_new_anchorless_ocrp_sr
if "EDSR" in paper_jangid_models:
    METHODS["EDSR"] = upsample_edsr_sr
if "QEDSR" in paper_jangid_models:
    METHODS["QEDSR"] = upsample_qedsr_sr
if "Q-RBSA-adapted" in paper_jangid_models:
    METHODS["Q-RBSA-adapted"] = upsample_qrbsa_adapted_sr
if "HAN" in paper_jangid_models:
    METHODS["HAN"] = upsample_han_sr
if "RCAN" in paper_jangid_models:
    METHODS["RCAN"] = upsample_rcan_sr
if "SAN" in paper_jangid_models:
    METHODS["SAN"] = upsample_san_sr
if model_atindama is not None:
    METHODS["Atindama inpainting"] = upsample_atindama_inpainting_sr

# Classical and geometric baselines retained for context.
METHODS["Nearest"] = upsample_nn
METHODS["Bicubic"] = upsample_bicubic
METHODS["Bicubic (F.interpolate)"] = upsample_bicubic_finterp
METHODS["SLERP"] = upsample_slerp
METHODS["Symm-SLERP"] = upsample_symm_slerp
METHODS["BA Sym-SLERP"] = upsample_ba_sym_slerp

print("Methods:", list(METHODS.keys()))


def run_all_methods(lr_q, hr_q, methods=METHODS, patch_rows=PATCH_ROWS, patch_cols=PATCH_COLS, sym_ops=sym, hr_grain_labels=HR_GRAIN_LABELS):
    hr_patch = hr_q[patch_rows, patch_cols]
    hr_patch_grain_labels = None if hr_grain_labels is None else np.asarray(hr_grain_labels[patch_rows, patch_cols])
    ipf_ref_dir = "Z"
    ipf_hr_patch = render_ipf_rgb(hr_patch, sym_ops, ref_dir=ipf_ref_dir)
    grod_hr, _ = compute_grod(hr_patch, grain_labels=hr_patch_grain_labels, sym_ops=sym_ops)
    kam_hr, _ = compute_kam(hr_patch, grain_labels=hr_patch_grain_labels, radius=1, sym_ops=sym_ops)
    boundary_hr = compute_boundary_mask(hr_q, sym_ops=sym_ops, threshold_deg=5.0, connectivity=4)
    max_mis_deg = max_misorientation_from_sym(sym_ops)
    mis_ref_patch = np.zeros(hr_patch.shape[:2], dtype=np.float64)
    mis_ref_full = np.zeros(hr_q.shape[:2], dtype=np.float64)
    hr_patch_stats = {
        "grod_mean": float(np.nanmean(grod_hr)),
        "grod_std": float(np.nanstd(grod_hr)),
        "kam_mean": float(np.nanmean(kam_hr)),
        "kam_std": float(np.nanstd(kam_hr)),
    }

    results = {}
    for name, fn in methods.items():
        print(f"Running {name}...", end=" ", flush=True)
        if device.type == "cuda":
            torch.cuda.empty_cache()
        try:
            sr_full = fn(lr_q, hr_q.shape[:2])
        except torch.OutOfMemoryError as exc:
            if device.type == "cuda":
                torch.cuda.empty_cache()
            print(f"skipped ({type(exc).__name__}: {exc})")
            continue
        except RuntimeError as exc:
            if "out of memory" in str(exc).lower():
                if device.type == "cuda":
                    torch.cuda.empty_cache()
                print(f"skipped ({type(exc).__name__}: {exc})")
                continue
            raise

        if sr_full.shape[:2] != hr_q.shape[:2]:
            print(f"skipped (shape mismatch: got {sr_full.shape[:2]}, expected {hr_q.shape[:2]})")
            continue

        sr_patch = sr_full[patch_rows, patch_cols]
        boundary_sr = compute_boundary_mask(sr_full, sym_ops=sym_ops, threshold_deg=5.0, connectivity=4)
        boundary_f1 = boundary_f1_score(boundary_sr, boundary_hr)
        grod, _ = compute_grod(sr_patch, grain_labels=hr_patch_grain_labels, sym_ops=sym_ops)
        kam, _ = compute_kam(sr_patch, grain_labels=hr_patch_grain_labels, radius=1, sym_ops=sym_ops)
        mis_patch = crystallographic_misorientation(sr_patch, hr_patch, sym_quats=sym_ops, degrees=True)
        mis_full = crystallographic_misorientation(sr_full, hr_q, sym_quats=sym_ops, degrees=True)
        ipf_sr_patch = render_ipf_rgb(sr_patch, sym_ops, ref_dir=ipf_ref_dir)

        grod_delta = grod - grod_hr
        grod_delta_std = float(np.nanstd(grod_delta))
        kam_delta = kam - kam_hr
        kam_delta_std = float(np.nanstd(kam_delta))

        mis_stats = {
            "mean": float(np.nanmean(mis_patch)),
            "median": float(np.nanmedian(mis_patch)),
            "p90": float(np.nanpercentile(mis_patch, 90)),
            "p95": float(np.nanpercentile(mis_patch, 95)),
            "p99": float(np.nanpercentile(mis_patch, 99)),
            "psnr": psnr_from_map(mis_ref_patch, mis_patch, max_val=max_mis_deg),
            "ssim": ssim_from_rgb_map(ipf_hr_patch, ipf_sr_patch, win_size=7, data_range=1.0),
        }
        mis_full_stats = {
            "mean": float(np.nanmean(mis_full)),
            "median": float(np.nanmedian(mis_full)),
            "p90": float(np.nanpercentile(mis_full, 90)),
            "p95": float(np.nanpercentile(mis_full, 95)),
            "p99": float(np.nanpercentile(mis_full, 99)),
            "psnr": psnr_from_map(mis_ref_full, mis_full, max_val=max_mis_deg),
            "frac_gt_5deg": float(np.nanmean(mis_full > 5.0)),
        }

        grod_mean = float(np.nanmean(grod))
        grod_std = float(np.nanstd(grod))
        grod_stats = {
            "mean": grod_mean,
            "std": grod_std,
            "hr_mean": hr_patch_stats["grod_mean"],
            "hr_std": hr_patch_stats["grod_std"],
            "mean_diff": float(grod_mean - hr_patch_stats["grod_mean"]),
            "abs_mean_diff": float(abs(grod_mean - hr_patch_stats["grod_mean"])),
            "abs_mean_mag_diff": float(abs(grod_mean) - abs(hr_patch_stats["grod_mean"])),
            "std_diff": float(grod_std - hr_patch_stats["grod_std"]),
            "abs_std_diff": float(abs(grod_std - hr_patch_stats["grod_std"])),
        }

        kam_mean = float(np.nanmean(kam))
        kam_std = float(np.nanstd(kam))
        kam_stats = {
            "mean": kam_mean,
            "std": kam_std,
            "hr_mean": hr_patch_stats["kam_mean"],
            "hr_std": hr_patch_stats["kam_std"],
            "mean_diff": float(kam_mean - hr_patch_stats["kam_mean"]),
            "abs_mean_diff": float(abs(kam_mean - hr_patch_stats["kam_mean"])),
            "abs_mean_mag_diff": float(abs(kam_mean) - abs(hr_patch_stats["kam_mean"])),
            "std_diff": float(kam_std - hr_patch_stats["kam_std"]),
            "abs_std_diff": float(abs(kam_std - hr_patch_stats["kam_std"])),
        }

        results[name] = {
            "sr_full": sr_full,
            "sr_patch": sr_patch,
            "mis_patch": mis_patch,
            "mis_full": mis_full,
            "mis_stats": mis_stats,
            "mis_full_stats": mis_full_stats,
            "boundary_f1": boundary_f1,
            "grod": grod,
            "grod_delta": grod_delta,
            "grod_delta_std": grod_delta_std,
            "grod_stats": grod_stats,
            "kam": kam,
            "kam_delta": kam_delta,
            "kam_delta_std": kam_delta_std,
            "kam_stats": kam_stats,
        }
        print(
            f"patch_mean={mis_stats['mean']:.3f}°, ",
            f"patch_p90/p95/p99={mis_stats['p90']:.3f}/{mis_stats['p95']:.3f}/{mis_stats['p99']:.3f}°, ",
            f"full_mean={mis_full_stats['mean']:.3f}°, ",
            f"full_p90/p95/p99={mis_full_stats['p90']:.3f}/{mis_full_stats['p95']:.3f}/{mis_full_stats['p99']:.3f}°, ",
            f"PSNR={mis_stats['psnr']:.2f} dB, ",
            f"SSIM(IPF)={mis_stats['ssim']:.4f}",
            f"BoundaryF1={boundary_f1:.4f}, ",
            f"|grodΔ|={grod_stats['abs_mean_diff']:.3f}°, ",
            f"|kamΔ|={kam_stats['abs_mean_diff']:.3f}°"
        )
        if device.type == "cuda":
            torch.cuda.empty_cache()

    if "Bicubic" in results and "Bicubic (F.interpolate)" in results:
        bic_cv = results["Bicubic"]["sr_patch"]
        bic_fi = results["Bicubic (F.interpolate)"]["sr_patch"]
        bic_backend_mis = crystallographic_misorientation(
            bic_cv,
            bic_fi,
            sym_quats=sym_ops,
            degrees=True,
        )
        bic_backend_summary = {
            "mean": float(np.nanmean(bic_backend_mis)),
            "median": float(np.nanmedian(bic_backend_mis)),
            "p90": float(np.nanpercentile(bic_backend_mis, 90)),
            "p95": float(np.nanpercentile(bic_backend_mis, 95)),
            "p99": float(np.nanpercentile(bic_backend_mis, 99)),
            "max": float(np.nanmax(bic_backend_mis)),
            "mean_abs_component_diff": float(np.mean(np.abs(bic_cv - bic_fi))),
        }
        results["__bicubic_backend_comparison__"] = {
            "mis_patch": bic_backend_mis,
            "summary": bic_backend_summary,
        }
        print(
            "OpenCV Bicubic vs F.interpolate Bicubic: "
            f"mean={bic_backend_summary['mean']:.4f}°, "
            f"p90/p95/p99={bic_backend_summary['p90']:.4f}/{bic_backend_summary['p95']:.4f}/{bic_backend_summary['p99']:.4f}°, "
            f"max={bic_backend_summary['max']:.4f}°, "
            f"mean_abs_component_diff={bic_backend_summary['mean_abs_component_diff']:.6f}"
        )

    results["__hr_patch__"] = hr_patch
    results["__hr_patch_grain_labels__"] = hr_patch_grain_labels
    results["__grod_hr__"] = grod_hr
    results["__kam_hr__"] = kam_hr
    results["__hr_patch_stats__"] = hr_patch_stats
    results["__max_mis_deg__"] = max_mis_deg
    return results


Methods: ['OCRP', 'EDSR', 'QEDSR', 'Q-RBSA-adapted', 'HAN', 'RCAN', 'SAN', 'Atindama inpainting', 'Nearest', 'Bicubic', 'Bicubic (F.interpolate)', 'SLERP', 'Symm-SLERP', 'BA Sym-SLERP']


In [10]:
_run_all_methods_original = run_all_methods


def run_all_methods(*args, **kwargs):
    try:
        return _run_all_methods_original(*args, **kwargs)
    except (ValueError, RuntimeError, AssertionError, FileNotFoundError) as exc:
        print(f"skipped ({type(exc).__name__}: {exc})")
        return {}


In [11]:
# ── Section 8: Run methods one-by-one on Open718 test block 0 ────────────────
# Set METHODS_TO_RUN to a subset like ["OCRP", "HAN", "Q-RBSA-adapted"] when you want to
# run one heavy method at a time and keep the accumulated outputs.
METHODS_TO_RUN = list(METHODS.keys())
results = {}
for method_name in METHODS_TO_RUN:
    partial_results = run_all_methods(
        lr,
        hr,
        methods=OrderedDict([(method_name, METHODS[method_name])]),
    )
    results.update(partial_results)
print("Done.")


Running OCRP... patch_mean=0.728°,  patch_p90/p95/p99=1.243/1.438/1.846°,  full_mean=2.411°,  full_p90/p95/p99=1.243/1.633/59.551°,  PSNR=46.80 dB,  SSIM(IPF)=0.7757 BoundaryF1=0.6646,  |grodΔ|=3.616°,  |kamΔ|=0.745°
Running EDSR... patch_mean=0.698°,  patch_p90/p95/p99=1.223/1.418/1.784°,  full_mean=2.577°,  full_p90/p95/p99=1.350/3.163/59.029°,  PSNR=47.02 dB,  SSIM(IPF)=0.7822 BoundaryF1=0.6506,  |grodΔ|=1.560°,  |kamΔ|=0.655°
Running QEDSR... patch_mean=0.704°,  patch_p90/p95/p99=1.247/1.446/1.808°,  full_mean=2.585°,  full_p90/p95/p99=1.331/3.345/59.270°,  PSNR=46.93 dB,  SSIM(IPF)=0.7798 BoundaryF1=0.6369,  |grodΔ|=0.897°,  |kamΔ|=0.630°
Running Q-RBSA-adapted... patch_mean=0.717°,  patch_p90/p95/p99=1.178/1.378/1.739°,  full_mean=2.856°,  full_p90/p95/p99=2.839/13.159/54.719°,  PSNR=46.90 dB,  SSIM(IPF)=0.7815 BoundaryF1=0.6877,  |grodΔ|=8.144°,  |kamΔ|=0.835°
Running HAN... patch_mean=0.672°,  patch_p90/p95/p99=1.152/1.351/1.718°,  full_mean=2.472°,  full_p90/p95/p99=1.678/4.84

/tmp/ipykernel_156364/3621659553.py:248: RuntimeWarning: Mean of empty slice
  kam = np.nanmean(vals, axis=0)


patch_mean=26.912°,  patch_p90/p95/p99=41.711/43.548/45.087°,  full_mean=11.014°,  full_p90/p95/p99=36.096/42.474/53.551°,  PSNR=15.71 dB,  SSIM(IPF)=0.0559 BoundaryF1=0.3042,  |grodΔ|=11.875°,  |kamΔ|=4.815°
Running Nearest... patch_mean=0.852°,  patch_p90/p95/p99=1.508/1.814/2.236°,  full_mean=4.200°,  full_p90/p95/p99=1.814/42.416/59.750°,  PSNR=44.64 dB,  SSIM(IPF)=0.7168 BoundaryF1=0.4433,  |grodΔ|=3.505°,  |kamΔ|=0.541°
Running Bicubic... patch_mean=14.142°,  patch_p90/p95/p99=35.347/39.371/44.173°,  full_mean=7.745°,  full_p90/p95/p99=27.424/39.248/56.579°,  PSNR=19.49 dB,  SSIM(IPF)=0.0666 BoundaryF1=0.3578,  |grodΔ|=0.247°,  |kamΔ|=4.573°
Running Bicubic (F.interpolate)... patch_mean=14.142°,  patch_p90/p95/p99=35.347/39.371/44.173°,  full_mean=7.745°,  full_p90/p95/p99=27.424/39.248/56.579°,  PSNR=19.49 dB,  SSIM(IPF)=0.0666 BoundaryF1=0.3578,  |grodΔ|=0.247°,  |kamΔ|=4.573°
Running SLERP... patch_mean=15.956°,  patch_p90/p95/p99=36.896/41.114/42.178°,  full_mean=7.488°,  ful

In [12]:
# ── Section 9b: Full-map IPF overview for all methods ───────────────────────
ref_dirs = ("X", "Y", "Z")
method_names = [name for name in METHODS.keys() if name in results]
col_labels = ["LR", "HR"] + method_names

full_maps = [
    dict(zip(ref_dirs, render_ipf_rgb(lr, sym))),
    dict(zip(ref_dirs, render_ipf_rgb(hr, sym))),
]
for name in method_names:
    full_maps.append(dict(zip(ref_dirs, render_ipf_rgb(results[name]["sr_full"], sym))))

# Save individual full-map IPF panels in a method/ref-dir hierarchy.
for label, ipf_map in zip(col_labels, full_maps):
    label_dir = FULL_MAP_PANEL_DIR / slugify_label(label)
    for rd in ref_dirs:
        save_rgb_panel(label_dir / f"ipf_{rd.lower()}.png", ipf_map[rd])
print("saved full-map IPF panel directory:", FULL_MAP_PANEL_DIR)

n_cols = len(col_labels)
full_map_dpi = 220
fig = plt.figure(figsize=(2.4 * n_cols, 9.8), dpi=full_map_dpi, constrained_layout=True)
gs = fig.add_gridspec(
    3,
    n_cols + 1,
    width_ratios=[0.34] + [1] * n_cols,
)
fig.suptitle(f"{sample_id} (4x4) — Full-map IPF Overview (All Methods)", fontsize=15)

for r, rd in enumerate(ref_dirs):
    ax_label = fig.add_subplot(gs[r, 0])
    ax_label.axis("off")
    ax_label.text(0.98, 0.5, f"Full IPF-{rd}", ha="right", va="center", fontsize=10)

    for c, ipf_map in enumerate(full_maps, start=1):
        ax = fig.add_subplot(gs[r, c])
        ax.imshow(ipf_map[rd])
        ax.axis("off")
        if r == 0:
            ax.set_title(col_labels[c - 1], fontsize=9, pad=6)

save_notebook_figure(fig, FULL_MAP_FIG_DIR / "full_map_ipf_xyz_overview.png", dpi=full_map_dpi)
plt.show()


saved full-map IPF panel directory: /data/home/umang/Materials/Reynolds-QSR_paper/analysis/out/patch_sample_comparison_all_methods_4x4/Open_718_Test_hr_x_block_0/full_map/ipf_xyz_panels
saved: /data/home/umang/Materials/Reynolds-QSR_paper/analysis/out/patch_sample_comparison_all_methods_4x4/Open_718_Test_hr_x_block_0/full_map/full_map_ipf_xyz_overview.png


<Figure size 8448x2156 with 51 Axes>

In [13]:
# ── Boundary F1 visualization: IPF-Z row + boundary masks ───────────────
from matplotlib.colors import Normalize

method_names = [name for name in METHODS.keys() if name in results]
col_labels = ["HR"] + method_names

# Row 1: IPF-Z
ipf_hr_z = render_ipf_rgb(hr, sym, ref_dir="Z")
ipf_sr_z = [render_ipf_rgb(results[name]["sr_full"], sym, ref_dir="Z") for name in method_names]

# Row 2: boundary masks (full map)
boundary_hr = compute_boundary_mask(hr, sym_ops=sym, threshold_deg=5.0, connectivity=4)
boundary_sr = [compute_boundary_mask(results[name]["sr_full"], sym_ops=sym, threshold_deg=5.0, connectivity=4) for name in method_names]

# Save full-map IPF-Z panels and boundary masks separately for reuse in paper/debug figures.
save_rgb_panel(BOUNDARY_FIG_DIR / "ipfz" / "hr_ipf_z.png", ipf_hr_z)
save_mask_panel(BOUNDARY_MASK_DIR / "hr_boundary_mask_5deg.png", boundary_hr)
for name, ipf_z, bmask in zip(method_names, ipf_sr_z, boundary_sr):
    slug = slugify_label(name)
    save_rgb_panel(BOUNDARY_FIG_DIR / "ipfz" / f"{slug}_ipf_z.png", ipf_z)
    save_mask_panel(BOUNDARY_MASK_DIR / f"{slug}_boundary_mask_5deg.png", bmask)
print("saved IPF-Z and boundary-mask directories:", BOUNDARY_FIG_DIR)

rows = 2
cols = len(col_labels)
fig = plt.figure(figsize=(2.6 * cols, 2.6 * rows), dpi=200)

# IPF-Z row
for j, label in enumerate(col_labels):
    ax = fig.add_subplot(rows, cols, j + 1)
    if label == "HR":
        ax.imshow(ipf_hr_z)
    else:
        ax.imshow(ipf_sr_z[j-1])
    ax.set_title(label, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])

# Boundary row
for j, label in enumerate(col_labels):
    ax = fig.add_subplot(rows, cols, cols + j + 1)
    if label == "HR":
        ax.imshow(boundary_hr, cmap="gray", vmin=0, vmax=1)
    else:
        ax.imshow(boundary_sr[j-1], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{label} boundaries", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle(f"{sample_id} (4x4) — IPF-Z and Boundary Masks (full map)", fontsize=12)
fig.tight_layout()
save_notebook_figure(fig, BOUNDARY_FIG_DIR / "full_map_ipfz_and_boundary_masks.png", dpi=220)
plt.show()


saved IPF-Z and boundary-mask directories: /data/home/umang/Materials/Reynolds-QSR_paper/analysis/out/patch_sample_comparison_all_methods_4x4/Open_718_Test_hr_x_block_0/boundary
saved: /data/home/umang/Materials/Reynolds-QSR_paper/analysis/out/patch_sample_comparison_all_methods_4x4/Open_718_Test_hr_x_block_0/boundary/full_map_ipfz_and_boundary_masks.png


<Figure size 7800x1040 with 30 Axes>

In [14]:
# ── Whole-map boundary F1 comparison ───────────────────────────────────────────
from IPython.display import display

boundary_threshold_deg = 5.0
boundary_connectivity = 4

method_names = [name for name in METHODS.keys() if name in results]

boundary_hr_full = compute_boundary_mask(
    hr,
    sym_ops=sym,
    threshold_deg=boundary_threshold_deg,
    connectivity=boundary_connectivity,
)

rows = []
for name in method_names:
    boundary_sr_full = compute_boundary_mask(
        results[name]["sr_full"],
        sym_ops=sym,
        threshold_deg=boundary_threshold_deg,
        connectivity=boundary_connectivity,
    )

    tp = int(np.sum(boundary_sr_full & boundary_hr_full))
    fp = int(np.sum(boundary_sr_full & ~boundary_hr_full))
    fn = int(np.sum(~boundary_sr_full & boundary_hr_full))

    precision = float(tp / (tp + fp)) if (tp + fp) > 0 else np.nan
    recall = float(tp / (tp + fn)) if (tp + fn) > 0 else np.nan
    f1 = boundary_f1_score(boundary_sr_full, boundary_hr_full)

    rows.append(
        {
            "Method": name,
            "Boundary Precision": precision,
            "Boundary Recall": recall,
            "Boundary F1": f1,
            "SR boundary %": 100.0 * float(np.mean(boundary_sr_full)),
            "HR boundary %": 100.0 * float(np.mean(boundary_hr_full)),
            "TP": tp,
            "FP": fp,
            "FN": fn,
        }
    )

df_boundary_full = (
    pd.DataFrame(rows)
    .set_index("Method")
    .sort_values("Boundary F1", ascending=False)
)
boundary_metrics_csv = BOUNDARY_FIG_DIR / "whole_map_boundary_metrics.csv"
df_boundary_full.to_csv(boundary_metrics_csv)
print("saved boundary metrics CSV:", boundary_metrics_csv)

fig, ax = plt.subplots(figsize=(max(8.5, 0.62 * len(df_boundary_full.index)), 4.5), dpi=180)
ax.bar(df_boundary_full.index, df_boundary_full["Boundary F1"].to_numpy(dtype=float), color="#4c78a8")
ax.set_ylim(0.0, 1.0)
ax.set_ylabel("Boundary F1")
ax.set_title(f"{sample_id} (4x4) — whole-map boundary F1 comparison")
ax.grid(True, axis="y", alpha=0.3)
ax.tick_params(axis="x", rotation=35, labelsize=8)
for label in ax.get_xticklabels():
    label.set_ha("right")
fig.tight_layout()
save_notebook_figure(fig, BOUNDARY_FIG_DIR / "whole_map_boundary_f1_comparison.png", dpi=220)
plt.show()


display(
    df_boundary_full.style
    .format(
        {
            "Boundary Precision": "{:.4f}",
            "Boundary Recall": "{:.4f}",
            "Boundary F1": "{:.4f}",
            "SR boundary %": "{:.2f}",
            "HR boundary %": "{:.2f}",
        }
    )
    .highlight_max(subset=["Boundary F1"], color="#e8f5e9")
    .set_caption(
        f"{sample_id} (4x4) — whole-map boundary metrics "
        f"(threshold={boundary_threshold_deg:.1f}°, connectivity={boundary_connectivity})"
    )
)

df_boundary_full


saved boundary metrics CSV: /data/home/umang/Materials/Reynolds-QSR_paper/analysis/out/patch_sample_comparison_all_methods_4x4/Open_718_Test_hr_x_block_0/boundary/whole_map_boundary_metrics.csv
saved: /data/home/umang/Materials/Reynolds-QSR_paper/analysis/out/patch_sample_comparison_all_methods_4x4/Open_718_Test_hr_x_block_0/boundary/whole_map_boundary_f1_comparison.png


<Figure size 1562.4x810 with 1 Axes>

,Boundary Precision,Boundary Recall,Boundary F1,SR boundary %,HR boundary %,TP,FP,FN
Method,,,,,,,,
HAN,0.6633,0.7410,0.7000,10.38,9.29,8011,4066,2800
Q-RBSA-adapted,0.5883,0.8275,0.6877,13.06,9.29,8946,6261,1865
OCRP,0.6981,0.6341,0.6646,8.44,9.29,6855,2964,3956
EDSR,0.6195,0.6850,0.6506,10.27,9.29,7405,4549,3406
RCAN,0.6246,0.6778,0.6501,10.08,9.29,7328,4405,3483
QEDSR,0.6102,0.6660,0.6369,10.14,9.29,7200,4600,3611
SAN,0.6414,0.6293,0.6353,9.11,9.29,6803,3804,4008
Symm-SLERP,0.3476,0.8273,0.4896,22.10,9.29,8944,16784,1867
Nearest,0.4153,0.4753,0.4433,10.63,9.29,5138,7234,5673


,Boundary Precision,Boundary Recall,Boundary F1,SR boundary %,HR boundary %,TP,FP,FN
Method,,,,,,,,
HAN,0.663327,0.741005,0.700017,10.375430,9.287801,8011,4066,2800
Q-RBSA-adapted,0.588282,0.827491,0.687678,13.064433,9.287801,8946,6261,1865
OCRP,0.698136,0.634076,0.664566,8.435567,9.287801,6855,2964,3956
EDSR,0.619458,0.684951,0.650560,10.269759,9.287801,7405,4549,3406
RCAN,0.624563,0.677828,0.650106,10.079897,9.287801,7328,4405,3483
QEDSR,0.610169,0.665988,0.636858,10.137457,9.287801,7200,4600,3611
SAN,0.641369,0.629266,0.635260,9.112543,9.287801,6803,3804,4008
Symm-SLERP,0.347637,0.827306,0.489559,22.103093,9.287801,8944,16784,1867
Nearest,0.415293,0.475257,0.443256,10.628866,9.287801,5138,7234,5673


In [15]:
# ── Section 10: Combined patch dashboard (methods = columns, metrics = rows) ──
from matplotlib.colors import Normalize

method_names = [name for name in METHODS.keys() if name in results]
ref_dirs = ("X", "Y", "Z")
n_methods = len(method_names)
panel_names = ["HR Patch"] + method_names
n_data_cols = len(panel_names)

hr_patch = results["__hr_patch__"]
grod_hr = results["__grod_hr__"]
kam_hr = results["__kam_hr__"]

patch_ipf = {"HR Patch": dict(zip(ref_dirs, render_ipf_rgb(hr_patch, sym)))}
for name in method_names:
    patch_ipf[name] = dict(zip(ref_dirs, render_ipf_rgb(results[name]["sr_patch"], sym)))

vmax_grod = max([np.nanmax(grod_hr)] + [np.nanmax(results[n]["grod"]) for n in method_names])
vmax_kam = max([np.nanmax(kam_hr)] + [np.nanmax(results[n]["kam"]) for n in method_names])

row_labels = [
    "Patch IPF-X",
    "Patch IPF-Y",
    "Patch IPF-Z",
    "Patch intra-grain\nGROD",
    "Patch intra-grain\nKAM",
    "Patch Misorientation\nHistogram",
]

fig = plt.figure(
    figsize=(2.2 * (n_data_cols + 1.45), 15.3),
    constrained_layout=True,
)
gs = fig.add_gridspec(
    6,
    n_data_cols + 2,
    width_ratios=[0.48] + [1] * n_data_cols + [0.08],
    height_ratios=[1, 1, 1, 1, 1, 1.24],
)
fig.suptitle(f"{sample_id} (4x4) — Blue-grain Patch Comparison Dashboard", fontsize=17)

for r, label in enumerate(row_labels):
    ax_label = fig.add_subplot(gs[r, 0])
    ax_label.axis("off")
    ax_label.text(0.98, 0.5, label, ha="right", va="center", fontsize=9)

axes = []
for r in range(5):
    row_axes = []
    for c, panel_name in enumerate(panel_names, start=1):
        ax = fig.add_subplot(gs[r, c])
        ax.axis("off")
        if r == 0:
            ax.set_title(panel_name, fontsize=8, pad=4)
        row_axes.append(ax)
    axes.append(row_axes)

for ridx, rd in enumerate(ref_dirs):
    axes[ridx][0].imshow(patch_ipf["HR Patch"][rd])
    for c, name in enumerate(method_names, start=1):
        axes[ridx][c].imshow(patch_ipf[name][rd])

norm_grod = Normalize(vmin=0, vmax=vmax_grod)
for c in range(n_data_cols):
    data = grod_hr if c == 0 else results[method_names[c - 1]]["grod"]
    im_grod = axes[3][c].imshow(data, cmap="inferno", norm=norm_grod)
grod_cax = fig.add_subplot(gs[3, -1])
fig.colorbar(im_grod, cax=grod_cax, label="GROD (°)")

norm_kam = Normalize(vmin=0, vmax=vmax_kam)
for c in range(n_data_cols):
    data = kam_hr if c == 0 else results[method_names[c - 1]]["kam"]
    im_kam = axes[4][c].imshow(data, cmap="coolwarm", norm=norm_kam)
kam_cax = fig.add_subplot(gs[4, -1])
fig.colorbar(im_kam, cax=kam_cax, label="KAM (°), blue → red")

ax_hist_ref = fig.add_subplot(gs[5, 1])
ax_hist_ref.axis("off")
ax_hist_ref.text(0.5, 0.57, "HR patch\nreference", ha="center", va="center", fontsize=9, color="0.35")
ax_hist_ref.text(0.5, 0.29, "No histogram here because\nmisorientation is defined vs HR.", ha="center", va="center", fontsize=7.5, color="0.45")

hist_gs = gs[5, 2:n_data_cols + 1].subgridspec(1, n_methods, wspace=0.14)
hist_colors = list(plt.cm.tab10.colors) + list(plt.cm.Set2.colors)
for idx, name in enumerate(method_names):
    ax = fig.add_subplot(hist_gs[0, idx])
    m = results[name]["mis_patch"].ravel()
    m = m[~np.isnan(m)]
    m_max = float(np.nanmax(m)) if m.size else 1.0
    x_hi = max(1.0, m_max * 1.02)
    bins = np.linspace(0.0, x_hi, 41)
    ax.hist(
        m,
        bins=bins,
        density=True,
        color=hist_colors[idx % len(hist_colors)],
        alpha=0.84,
    )
    ax.axvline(np.nanmean(m), color="black", linestyle="--", linewidth=0.8)
    ax.grid(True, alpha=0.25)
    ax.set_xlim(0, x_hi)
    ax.tick_params(labelsize=7)
    ax.set_title(name, fontsize=7, pad=2)
    ax.set_xlabel("deg", fontsize=7)
    ax.text(
        0.98,
        0.94,
        f"max={m_max:.2f}°",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=6.5,
        color="0.35",
    )
    if idx == 0:
        ax.set_ylabel("density", fontsize=7)
    else:
        ax.set_yticklabels([])

save_notebook_figure(fig, PATCH_FIG_DIR / "patch_dashboard_basic_ipf_grod_kam_hist.png", dpi=220)
plt.show()


saved: /data/home/umang/Materials/Reynolds-QSR_paper/analysis/out/patch_sample_comparison_all_methods_4x4/Open_718_Test_hr_x_block_0/patch/patch_dashboard_basic_ipf_grod_kam_hist.png


<Figure size 3619x1530 with 98 Axes>

In [16]:
# ── Section 11: Combined patch dashboard (methods = columns, metrics = rows) ──
from matplotlib.colors import Normalize, TwoSlopeNorm

method_names = [name for name in METHODS.keys() if name in results]
ref_dirs = ("X", "Y", "Z")
n_methods = len(method_names)
panel_names = ["HR Patch"] + method_names
n_data_cols = len(panel_names)

hr_patch = results["__hr_patch__"]
grod_hr = results["__grod_hr__"]
kam_hr = results["__kam_hr__"]

patch_ipf = {"HR Patch": dict(zip(ref_dirs, render_ipf_rgb(hr_patch, sym)))}
for name in method_names:
    patch_ipf[name] = dict(zip(ref_dirs, render_ipf_rgb(results[name]["sr_patch"], sym)))

vmax_grod = max([np.nanmax(grod_hr)] + [np.nanmax(results[n]["grod"]) for n in method_names])
vmax_kam = max([np.nanmax(kam_hr)] + [np.nanmax(results[n]["kam"]) for n in method_names])
vmax_grod_delta = max(
    [0.0] + [float(np.nanmax(np.abs(results[n]["grod_delta"]))) for n in method_names]
 )
KAM_DELTA_PLOT_CLIP_DEG = 3.0

row_labels = [
    "Patch IPF-X",
    "Patch IPF-Y",
    "Patch IPF-Z",
    "Patch intra-grain\nGROD",
    "Patch Delta intra-grain\nGROD (model - HR)",
    "Patch intra-grain\nKAM",
    "Patch Delta intra-grain\nKAM (model - HR)",
    "Patch Misorientation\nHistogram",
]

fig = plt.figure(
    figsize=(2.2 * (n_data_cols + 1.45), 19.5),
    constrained_layout=True,
)
gs = fig.add_gridspec(
    8,
    n_data_cols + 2,
    width_ratios=[0.48] + [1] * n_data_cols + [0.08],
    height_ratios=[1, 1, 1, 1, 1, 1, 1, 1.24],
)
fig.suptitle(f"{sample_id} (4x4) — Blue-grain Patch Comparison Dashboard", fontsize=17)

for r, label in enumerate(row_labels):
    ax_label = fig.add_subplot(gs[r, 0])
    ax_label.axis("off")
    ax_label.text(0.98, 0.5, label, ha="right", va="center", fontsize=9)

axes = []
for r in range(7):
    row_axes = []
    for c, panel_name in enumerate(panel_names, start=1):
        ax = fig.add_subplot(gs[r, c])
        ax.axis("off")
        if r == 0:
            ax.set_title(panel_name, fontsize=8, pad=4)
        row_axes.append(ax)
    axes.append(row_axes)

for ridx, rd in enumerate(ref_dirs):
    axes[ridx][0].imshow(patch_ipf["HR Patch"][rd])
    for c, name in enumerate(method_names, start=1):
        axes[ridx][c].imshow(patch_ipf[name][rd])

norm_grod = Normalize(vmin=0, vmax=vmax_grod)
for c in range(n_data_cols):
    data = grod_hr if c == 0 else results[method_names[c - 1]]["grod"]
    im_grod = axes[3][c].imshow(data, cmap="inferno", norm=norm_grod)
grod_cax = fig.add_subplot(gs[3, -1])
fig.colorbar(im_grod, cax=grod_cax, label="GROD (°)")

norm_grod_delta = TwoSlopeNorm(vmin=-vmax_grod_delta, vcenter=0.0, vmax=vmax_grod_delta)
for c in range(n_data_cols):
    data = np.zeros_like(grod_hr) if c == 0 else results[method_names[c - 1]]["grod_delta"]
    im_grod_delta = axes[4][c].imshow(data, cmap="RdBu_r", norm=norm_grod_delta)
grod_delta_cax = fig.add_subplot(gs[4, -1])
fig.colorbar(im_grod_delta, cax=grod_delta_cax, label="ΔGROD (°), model - HR")

norm_kam = Normalize(vmin=0, vmax=vmax_kam)
for c in range(n_data_cols):
    data = kam_hr if c == 0 else results[method_names[c - 1]]["kam"]
    im_kam = axes[5][c].imshow(data, cmap="coolwarm", norm=norm_kam)
kam_cax = fig.add_subplot(gs[5, -1])
fig.colorbar(im_kam, cax=kam_cax, label="KAM (°), blue → red")

norm_kam_delta = TwoSlopeNorm(
    vmin=-KAM_DELTA_PLOT_CLIP_DEG,
    vcenter=0.0,
    vmax=KAM_DELTA_PLOT_CLIP_DEG,
)
for c in range(n_data_cols):
    data = (
        np.zeros_like(kam_hr)
        if c == 0
        else np.clip(
            results[method_names[c - 1]]["kam_delta"],
            -KAM_DELTA_PLOT_CLIP_DEG,
            KAM_DELTA_PLOT_CLIP_DEG,
        )
    )
    im_kam_delta = axes[6][c].imshow(data, cmap="RdBu_r", norm=norm_kam_delta)
kam_delta_cax = fig.add_subplot(gs[6, -1])
fig.colorbar(
    im_kam_delta,
    cax=kam_delta_cax,
    label=f"ΔKAM (°), model - HR (clipped at ±{KAM_DELTA_PLOT_CLIP_DEG:.0f}°)",
)

ax_hist_ref = fig.add_subplot(gs[7, 1])
ax_hist_ref.axis("off")
ax_hist_ref.text(0.5, 0.57, "HR patch\nreference", ha="center", va="center", fontsize=9, color="0.35")
ax_hist_ref.text(0.5, 0.29, "No histogram here because\nmisorientation is defined vs HR.", ha="center", va="center", fontsize=7.5, color="0.45")

hist_gs = gs[7, 2:n_data_cols + 1].subgridspec(1, n_methods, wspace=0.14)
hist_colors = list(plt.cm.tab10.colors) + list(plt.cm.Set2.colors)
for idx, name in enumerate(method_names):
    ax = fig.add_subplot(hist_gs[0, idx])
    m = results[name]["mis_patch"].ravel()
    m = m[~np.isnan(m)]
    m_max = float(np.nanmax(m)) if m.size else 1.0
    x_hi = max(1.0, m_max * 1.02)
    bins = np.linspace(0.0, x_hi, 41)
    ax.hist(
        m,
        bins=bins,
        density=True,
        color=hist_colors[idx % len(hist_colors)],
        alpha=0.84,
    )
    ax.axvline(np.nanmean(m), color="black", linestyle="--", linewidth=0.8)
    ax.grid(True, alpha=0.25)
    ax.set_xlim(0, x_hi)
    ax.tick_params(labelsize=7)
    ax.set_title(name, fontsize=7, pad=2)
    ax.set_xlabel("deg", fontsize=7)
    ax.text(
        0.98,
        0.94,
        f"max={m_max:.2f}°",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=6.5,
        color="0.35",
    )
    if idx == 0:
        ax.set_ylabel("density", fontsize=7)
    else:
        ax.set_yticklabels([])

save_notebook_figure(fig, PATCH_FIG_DIR / "patch_dashboard_with_grod_kam_deltas.png", dpi=220)
plt.show()


saved: /data/home/umang/Materials/Reynolds-QSR_paper/analysis/out/patch_sample_comparison_all_methods_4x4/Open_718_Test_hr_x_block_0/patch/patch_dashboard_with_grod_kam_deltas.png


<Figure size 3619x1950 with 132 Axes>

In [17]:
# ── Section 12: Patch scalar metrics summary ─────────────────────────────────
from IPython.display import display

method_names = [name for name in METHODS.keys() if name in results]

rows = []
for name in method_names:
    r = results[name]
    rows.append({
        "Method": name,
        "Patch Mis mean": r["mis_stats"]["mean"],
        "Patch Mis median": r["mis_stats"]["median"],
        "Patch Mis p90": r["mis_stats"]["p90"],
        "Patch Mis p95": r["mis_stats"]["p95"],
        "Patch Mis p99": r["mis_stats"]["p99"],
        "Patch PSNR mis": r["mis_stats"]["psnr"],
        "SSIM IPF": r["mis_stats"]["ssim"],
        "Full Mis mean": r["mis_full_stats"]["mean"],
        "Full Mis median": r["mis_full_stats"]["median"],
        "Full Mis p90": r["mis_full_stats"]["p90"],
        "Full Mis p95": r["mis_full_stats"]["p95"],
        "Full Mis p99": r["mis_full_stats"]["p99"],
        "Full PSNR mis": r["mis_full_stats"]["psnr"],
        "Full frac >5°": r["mis_full_stats"]["frac_gt_5deg"],
        "Boundary F1": r["boundary_f1"],
        "abs(mean(GROD method - GROD HR))": abs(r["grod_stats"]["mean_diff"]),
        "std(GROD method - GROD HR)": r["grod_delta_std"],
        "abs(mean(KAM method - KAM HR))": abs(r["kam_stats"]["mean_diff"]),
        "std(KAM method - KAM HR)": r["kam_delta_std"],
    })

df_summary = pd.DataFrame(rows).set_index("Method").T
metric_order = [
    "Patch Mis mean",
    "Patch Mis median",
    "Patch Mis p90",
    "Patch Mis p95",
    "Patch Mis p99",
    "Patch PSNR mis",
    "SSIM IPF",
    "Full Mis mean",
    "Full Mis median",
    "Full Mis p90",
    "Full Mis p95",
    "Full Mis p99",
    "Full PSNR mis",
    "Full frac >5°",
    "Boundary F1",
    "abs(mean(GROD method - GROD HR))",
    "std(GROD method - GROD HR)",
    "abs(mean(KAM method - KAM HR))",
    "std(KAM method - KAM HR)",
    
]
df_summary = df_summary.loc[metric_order]

metric_pref = {
    "Patch Mis mean": "min",
    "Patch Mis median": "min",
    "Patch Mis p90": "min",
    "Patch Mis p95": "min",
    "Patch Mis p99": "min",
    "Patch PSNR mis": "max",
    "SSIM IPF": "max",
    "Full Mis mean": "min",
    "Full Mis median": "min",
    "Full Mis p90": "min",
    "Full Mis p95": "min",
    "Full Mis p99": "min",
    "Full PSNR mis": "max",
    "Full frac >5°": "min",
    "Boundary F1": "max",
    "abs(mean(GROD method - GROD HR))": "min",
    "std(GROD method - GROD HR)": "min",
    "abs(mean(KAM method - KAM HR))": "min",
    "std(KAM method - KAM HR)": "min",
}

metrics_csv = METRICS_FIG_DIR / "patch_scalar_metrics_summary.csv"
df_summary.to_csv(metrics_csv)
print("saved metrics CSV:", metrics_csv)

percentile_metrics = ["Patch Mis p90", "Patch Mis p95", "Patch Mis p99"]
percentile_df = df_summary.loc[percentile_metrics].T.astype(float)
fig, ax = plt.subplots(figsize=(max(9.0, 0.72 * len(percentile_df.index)), 4.8), dpi=180)
x = np.arange(len(percentile_df.index))
bar_width = 0.24
for offset, metric, color in zip((-bar_width, 0.0, bar_width), percentile_metrics, ["#4c78a8", "#f58518", "#e45756"]):
    ax.bar(x + offset, percentile_df[metric].to_numpy(dtype=float), bar_width, label=metric.replace("Patch Mis ", ""), color=color)
ax.set_xticks(x)
ax.set_xticklabels(percentile_df.index, rotation=35, ha="right", fontsize=8)
ax.set_ylabel("crystallographic misorientation error (°)")
ax.set_title(f"{sample_id} (4x4) — patch tail-error percentiles")
ax.grid(True, axis="y", alpha=0.3)
ax.legend(title="Percentile")
fig.tight_layout()
save_notebook_figure(fig, METRICS_FIG_DIR / "patch_misorientation_percentiles_p90_p95_p99.png", dpi=220)
plt.show()

full_percentile_metrics = ["Full Mis p90", "Full Mis p95", "Full Mis p99"]
full_percentile_df = df_summary.loc[full_percentile_metrics].T.astype(float)
fig, ax = plt.subplots(figsize=(max(9.0, 0.72 * len(full_percentile_df.index)), 4.8), dpi=180)
x = np.arange(len(full_percentile_df.index))
for offset, metric, color in zip((-bar_width, 0.0, bar_width), full_percentile_metrics, ["#4c78a8", "#f58518", "#e45756"]):
    ax.bar(x + offset, full_percentile_df[metric].to_numpy(dtype=float), bar_width, label=metric.replace("Full Mis ", ""), color=color)
ax.set_xticks(x)
ax.set_xticklabels(full_percentile_df.index, rotation=35, ha="right", fontsize=8)
ax.set_ylabel("crystallographic misorientation error (°)")
ax.set_title(f"{sample_id} (4x4) — full-map tail-error percentiles")
ax.grid(True, axis="y", alpha=0.3)
ax.legend(title="Percentile")
fig.tight_layout()
save_notebook_figure(fig, METRICS_FIG_DIR / "full_map_misorientation_percentiles_p90_p95_p99.png", dpi=220)
plt.show()

score_rows = []
for metric in metric_order:
    vals = pd.to_numeric(df_summary.loc[metric], errors="coerce").to_numpy(dtype=float)
    direction = metric_pref[metric]
    if direction == "max":
        key_vals = -vals
    elif direction == "absmin":
        key_vals = np.abs(vals)
    else:
        key_vals = vals
    finite = np.isfinite(key_vals)
    row = np.full_like(key_vals, np.nan, dtype=float)
    if np.any(finite):
        lo = np.nanmin(key_vals[finite])
        hi = np.nanmax(key_vals[finite])
        row[finite] = 0.0 if np.isclose(lo, hi) else (key_vals[finite] - lo) / (hi - lo)
    score_rows.append(row)
score_matrix = np.vstack(score_rows)
fig, ax = plt.subplots(figsize=(max(9.5, 0.72 * len(method_names)), 0.48 * len(metric_order) + 2.0), dpi=180)
im = ax.imshow(score_matrix, cmap="RdYlGn_r", vmin=0.0, vmax=1.0, aspect="auto")
ax.set_xticks(np.arange(len(method_names)))
ax.set_xticklabels(method_names, rotation=35, ha="right", fontsize=8)
ax.set_yticks(np.arange(len(metric_order)))
ax.set_yticklabels(metric_order, fontsize=8)
ax.set_title(f"{sample_id} (4x4) — normalized patch metric scorecard")
for i, metric in enumerate(metric_order):
    for j, name in enumerate(method_names):
        value = df_summary.loc[metric, name]
        if pd.notna(value):
            ax.text(j, i, f"{float(value):.2f}", ha="center", va="center", fontsize=6.5, color="black")
cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label("normalized metric error (0 = best in row)")
fig.tight_layout()
save_notebook_figure(fig, METRICS_FIG_DIR / "patch_scalar_metrics_normalized_scorecard.png", dpi=220)
plt.show()


quick_rows = []
for metric in df_summary.index:
    vals = pd.to_numeric(df_summary.loc[metric], errors="coerce")
    direction = metric_pref.get(metric)
    if direction is None:
        print(f"Warning: metric_pref missing '{metric}', defaulting to 'min'")
        direction = 'min'
    if direction == "max":
        best_method = vals.idxmax()
        best_value = float(np.nanmax(vals.to_numpy(dtype=float)))
        better = "higher"
    elif direction == "absmin":
        best_method = np.abs(vals).idxmin()
        best_value = float(vals.loc[best_method])
        better = "closest to 0"
    else:
        best_method = vals.idxmin()
        best_value = float(np.nanmin(vals.to_numpy(dtype=float)))
        better = "lower"
    quick_rows.append({
        "Metric": metric,
        "Better": better,
        "Best method": best_method,
        "Best value": best_value,
    })
quick_read = pd.DataFrame(quick_rows).set_index("Metric")
quick_read_csv = METRICS_FIG_DIR / "quick_read_best_by_metric.csv"
quick_read.to_csv(quick_read_csv)
print("saved quick-read CSV:", quick_read_csv)

reference_means = pd.DataFrame(
    {
        "HR patch": {
            "GROD HR mean": results["__hr_patch_stats__"]["grod_mean"],
            "GROD HR std": results["__hr_patch_stats__"]["grod_std"],
            "KAM HR mean": results["__hr_patch_stats__"]["kam_mean"],
            "KAM HR std": results["__hr_patch_stats__"]["kam_std"],
        }
    }
)

print("Scalar summary: lower is better except PSNR mis and SSIM IPF, where higher is better.")
print("Patch Mis metrics are computed only on the selected large-blue-grain patch; Full Mis metrics are computed over the whole aligned 300×388 map.")
print("GROD/KAM are computed intra-grain using HR-derived grain labels cropped to the patch.")
print("|mean(GROD method - GROD HR)| and |mean(KAM method - KAM HR)| are absolute mean differences; lower is better.")
print("PSNR uses misorientation vs zero error; SSIM is computed on the IPF RGB patch (ref_dir=Z) vs HR.")
if not SKIMAGE_AVAILABLE:
    print("SSIM is unavailable because scikit-image is not installed in this kernel.")



grod_kam_means = pd.DataFrame(
    {
        "GROD mean": {name: results[name]["grod_stats"]["mean"] for name in method_names},
        "KAM mean": {name: results[name]["kam_stats"]["mean"] for name in method_names},
    }
)

grod_kam_means.loc["HR reference"] = [
    results["__hr_patch_stats__"]["grod_mean"],
    results["__hr_patch_stats__"]["kam_mean"],
]
display(
    reference_means.style
    .format("{:.3f}")
    .set_caption("HR patch reference means")
    .set_table_styles([
        {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "12px"), ("font-weight", "600")]},
        {"selector": "th", "props": [("text-align", "center"), ("padding", "6px 10px"), ("background-color", "#f4f6f8")]},
        {"selector": "td", "props": [("text-align", "center"), ("padding", "6px 10px")]},
    ])
)



display(
    grod_kam_means.style
    .format("{:.3f}")
    .set_caption("Intra-grain GROD/KAM means by method (HR reference included)")
    .set_table_styles([
        {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "12px"), ("font-weight", "600")]},
        {"selector": "th", "props": [("text-align", "center"), ("padding", "6px 10px"), ("background-color", "#f4f6f8")]},
        {"selector": "td", "props": [("text-align", "center"), ("padding", "6px 10px")]},
    ])
)

display(
    quick_read.style
    .format({"Best value": "{:.3f}"})
    .set_caption("Quick read: best-performing method by metric")
    .set_table_styles([
        {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "12px"), ("font-weight", "600")]},
        {"selector": "th", "props": [("text-align", "center"), ("padding", "6px 10px"), ("background-color", "#f4f6f8")]},
        {"selector": "td", "props": [("text-align", "center"), ("padding", "6px 10px")]},
    ])
)


def _highlight_by_direction(row):
    vals = row.to_numpy(dtype=float)
    finite = np.isfinite(vals)
    if not np.any(finite):
        return [""] * len(vals)
    direction = metric_pref[row.name]
    if direction == "max":
        best = np.nanmax(vals)
        worst = np.nanmin(vals)
        key_vals = vals
    elif direction == "absmin":
        key_vals = np.abs(vals)
        best = np.nanmin(key_vals)
        worst = np.nanmax(key_vals)
    else:
        best = np.nanmin(vals)
        worst = np.nanmax(vals)
        key_vals = vals

    styles = []
    for i, v in enumerate(vals):
        if not np.isfinite(v):
            styles.append("color: #777777;")
        elif np.isclose(key_vals[i], best, equal_nan=False):
            styles.append("background-color: #e8f5e9; font-weight: 700; border: 2px solid #2e7d32;")
        elif np.isclose(key_vals[i], worst, equal_nan=False) and np.count_nonzero(finite) > 1:
            styles.append("background-color: #fdecea;")
        else:
            styles.append("")
    return styles


styled_summary = (
    df_summary.style
    .format("{:.3f}")
    .apply(_highlight_by_direction, axis=1)
    .set_caption("Patch scalar metrics summary")
    .set_table_styles([
        {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "13px"), ("font-weight", "600")]},
        {"selector": "th", "props": [("text-align", "center"), ("padding", "6px 10px"), ("background-color", "#f4f6f8")]},
        {"selector": "td", "props": [("text-align", "center"), ("padding", "6px 10px")]},
    ])
)

display(styled_summary)

df_summary


saved metrics CSV: /data/home/umang/Materials/Reynolds-QSR_paper/analysis/out/patch_sample_comparison_all_methods_4x4/Open_718_Test_hr_x_block_0/metrics/patch_scalar_metrics_summary.csv
saved: /data/home/umang/Materials/Reynolds-QSR_paper/analysis/out/patch_sample_comparison_all_methods_4x4/Open_718_Test_hr_x_block_0/metrics/patch_misorientation_percentiles_p90_p95_p99.png


<Figure size 1814.4x864 with 1 Axes>

saved: /data/home/umang/Materials/Reynolds-QSR_paper/analysis/out/patch_sample_comparison_all_methods_4x4/Open_718_Test_hr_x_block_0/metrics/full_map_misorientation_percentiles_p90_p95_p99.png


<Figure size 1814.4x864 with 1 Axes>

saved: /data/home/umang/Materials/Reynolds-QSR_paper/analysis/out/patch_sample_comparison_all_methods_4x4/Open_718_Test_hr_x_block_0/metrics/patch_scalar_metrics_normalized_scorecard.png


...

,HR patch
GROD HR mean,27.655
GROD HR std,0.486
KAM HR mean,0.875
KAM HR std,0.381


,GROD mean,KAM mean
OCRP,31.272,0.130
EDSR,29.215,0.219
QEDSR,28.553,0.245
Q-RBSA-adapted,35.799,0.040
HAN,33.006,0.136
RCAN,30.355,0.236
SAN,30.024,0.244
Atindama inpainting,15.781,5.690
Nearest,31.160,0.334
Bicubic,27.408,5.448


,Better,Best method,Best value
Metric,,,
Patch Mis mean,lower,HAN,0.672
Patch Mis median,lower,HAN,0.619
Patch Mis p90,lower,HAN,1.152
Patch Mis p95,lower,HAN,1.351
Patch Mis p99,lower,HAN,1.718
Patch PSNR mis,higher,HAN,47.392
SSIM IPF,higher,HAN,0.792
Full Mis mean,lower,OCRP,2.411
Full Mis median,lower,SAN,0.600


Method,OCRP,EDSR,QEDSR,Q-RBSA-adapted,HAN,RCAN,SAN,Atindama inpainting,Nearest,Bicubic,Bicubic (F.interpolate),SLERP,Symm-SLERP,BA Sym-SLERP
Patch Mis mean,0.728,0.698,0.704,0.717,0.672,0.696,0.706,26.912,0.852,14.142,14.142,15.956,0.783,0.783
Patch Mis median,0.675,0.666,0.663,0.718,0.619,0.656,0.649,28.754,1.000,9.178,9.178,11.228,0.769,0.769
Patch Mis p90,1.243,1.223,1.247,1.178,1.152,1.224,1.232,41.711,1.508,35.347,35.347,36.896,1.347,1.347
Patch Mis p95,1.438,1.418,1.446,1.378,1.351,1.440,1.423,43.548,1.814,39.371,39.371,41.114,1.578,1.578
Patch Mis p99,1.846,1.784,1.808,1.739,1.718,1.777,1.763,45.087,2.236,44.173,44.173,42.178,1.986,1.986
Patch PSNR mis,46.796,47.023,46.930,46.904,47.392,47.047,47.000,15.714,44.638,19.488,19.488,18.440,46.027,46.027
SSIM IPF,0.776,0.782,0.780,0.781,0.792,0.784,0.783,0.056,0.717,0.067,0.067,0.076,0.754,0.754
Full Mis mean,2.411,2.577,2.585,2.856,2.472,2.546,2.519,11.014,4.200,7.745,7.745,7.488,4.640,5.929
Full Mis median,0.612,0.611,0.605,0.758,0.673,0.611,0.600,3.411,1.000,1.507,1.507,0.955,0.853,0.761
Full Mis p90,1.243,1.350,1.331,2.839,1.678,1.331,1.271,36.096,1.814,27.424,27.424,30.308,16.074,25.634


Method,OCRP,EDSR,QEDSR,Q-RBSA-adapted,HAN,RCAN,SAN,Atindama inpainting,Nearest,Bicubic,Bicubic (F.interpolate),SLERP,Symm-SLERP,BA Sym-SLERP
Patch Mis mean,0.727943,0.697518,0.704099,0.716597,0.671834,0.696319,0.705952,26.912375,0.851730,14.142022,14.142022,15.955545,0.782687,0.782687
Patch Mis median,0.674864,0.666001,0.663070,0.718362,0.618808,0.656129,0.648520,28.753939,0.999955,9.177860,9.177858,11.227748,0.769468,0.769465
Patch Mis p90,1.242523,1.223493,1.246791,1.177982,1.152236,1.224299,1.232167,41.711406,1.508324,35.346823,35.346824,36.895548,1.346899,1.346899
Patch Mis p95,1.437767,1.418128,1.446381,1.377608,1.351281,1.440463,1.423163,43.548232,1.813602,39.370582,39.370580,41.114317,1.578175,1.578173
Patch Mis p99,1.845940,1.784420,1.808231,1.739378,1.718014,1.776692,1.762641,45.087382,2.236048,44.173102,44.173101,42.178085,1.985756,1.985753
Patch PSNR mis,46.796483,47.022583,46.930196,46.903588,47.391553,47.047002,46.999831,15.713637,44.637668,19.488331,19.488331,18.440254,46.027417,46.027418
SSIM IPF,0.775670,0.782231,0.779807,0.781477,0.791997,0.784059,0.783204,0.055914,0.716750,0.066599,0.066599,0.076100,0.754361,0.754361
Full Mis mean,2.411114,2.577151,2.585296,2.855914,2.472326,2.545790,2.518690,11.013590,4.199525,7.744971,7.744971,7.487694,4.639816,5.928892
Full Mis median,0.611915,0.610970,0.605470,0.757605,0.673281,0.611265,0.600263,3.411097,0.999993,1.507473,1.507473,0.955093,0.853084,0.760869
Full Mis p90,1.242953,1.349748,1.331157,2.838785,1.677545,1.330986,1.270951,36.096485,1.813602,27.423509,27.423511,30.307579,16.074267,25.633648


In [18]:
# ── Final cleanup: release GPU memory ─────────────────────────────────────
import gc

_moved_models = []
_deleted_names = []

for _name, _obj in list(globals().items()):
    if isinstance(_obj, torch.nn.Module):
        try:
            _obj.to(torch.device('cpu'))
            _moved_models.append(_name)
        except Exception:
            pass

for _name, _obj in list(globals().items()):
    if isinstance(_obj, torch.Tensor) and _obj.is_cuda:
        del globals()[_name]
        _deleted_names.append(_name)

for _name in [
    'results', 'full_maps', 'patch_ipf', 'df_summary', 'styled_summary',
    'sr_t', 'lr_t', 'hr_t', 'feat_lr', 'feat_hr_flat', 'sr_flat',
    'lr_active', 'sr_active', 'sr_passive', 'debug_payload'
]:
    if _name in globals():
        del globals()[_name]
        _deleted_names.append(_name)

plt.close('all')
gc.collect()
if torch.cuda.is_available():
    try:
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    except Exception:
        pass

print('Moved models to CPU:', sorted(set(_moved_models)))
print('Deleted GPU-heavy globals:', sorted(set(_deleted_names)))
print('CUDA cleanup complete.')


Moved models to CPU: ['model_atindama', 'model_bicubic_finterp', 'model_local', 'model_new_anchorless_ocrp']
Deleted GPU-heavy globals: ['df_summary', 'full_maps', 'patch_ipf', 'results', 'styled_summary']
CUDA cleanup complete.
